# Observatório do Clima — Notebook Consolidado

**Grande Desafio DataConnect Solutions**

Este arquivo reúne os notebooks do repositório em uma sequência única, mantendo as células originais em seções identificadas. Versões alternativas são preservadas para auditoria e comparação; por isso, algumas etapas podem repetir operações.

## Instruções de execução
- Ambiente recomendado: Google Colab ou Jupyter com Python 3.
- Disponibilize as bases originais e os arquivos intermediários esperados pelas etapas.
- Execute as seções na ordem apresentada e confira os caminhos de entrada/saída nas células originais.
- Células que dependem de uploads, Google Drive ou resultados gerados em outra etapa podem exigir ajuste manual.
- A consolidação é estrutural; o pipeline completo não foi executado porque os arquivos de dados não foram fornecidos nesta operação.

## Sequência
1. ETL SINISA
2. Tratamento CAPAG
3. Exploração Atlas/S2iD
4. Cruzamento das bases
5. Transformação e filtros (duas versões preservadas)
6. Ranking, funil e robustez
7. Painel final
8. Validação independente
9. Análise exploratória adicional

In [ ]:
# Configuração comum do ambiente
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PASTA_PROJETO = Path(os.environ.get("PASTA_PROJETO", ".")).resolve()
PASTA_DADOS = Path(os.environ.get("PASTA_DADOS", str(PASTA_PROJETO / "data"))).resolve()
PASTA_SAIDA = Path(os.environ.get("PASTA_SAIDA", str(PASTA_PROJETO / "outputs"))).resolve()
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print("Python:", sys.version.split()[0])
print("Projeto:", PASTA_PROJETO)
print("Dados:", PASTA_DADOS)
print("Saídas:", PASTA_SAIDA)
print("Nota: as células originais podem usar caminhos específicos próprios; confira-os antes de executar.")

---

# 1. ETL SINISA — diagnóstico, limpeza e exportação

**Notebook de origem:** `Notebook/ETL_SINISA.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

# ETL SINISA – Diagnóstico e Limpeza

**Squad Observatório do Clima – Grande Desafio DataConnect Solutions**

Este notebook é o script completo de ETL (extração, padronização e exportação)
que localiza, limpa e junta, a partir dos três arquivos brutos do SINISA
(Ministério das Cidades), as informações que a squad usa no painel de
priorização de municípios do Carlos (Sedec/MIDR):

1. Quantidade de domicílios em área de risco de inundação (arquivo 1)
2. Se existe mapeamento de área de risco (arquivo 1)
3. Se o município é classificado como "crítico" (arquivo 2)
4. Percentual de vias com rede de drenagem (arquivo 2)
5. Investimento per capita em drenagem (arquivo 2)

O notebook segue quatro partes, nesta ordem: **(1)** diagnóstico dinâmico de
aba/linha/coluna nos 3 arquivos, cruzando os códigos com o glossário em PDF;
**(2)** limpeza estrutural — extração das colunas confirmadas, padronização
do `Cod_IBGE_7`, conversão de tipos e junção dos dois arquivos por município;
**(3)** documentação (log de transformações); **(4)** exportação do
`sinisa_limpo.csv`.

**Antes de rodar:** tenha em mãos os 3 arquivos originais:
- `SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx`
- `SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx`
- `GLOSSARIO_INFORMACOES_SINISA_AGUASPLUVIAIS.pdf`

## 0. Configuração do ambiente e upload dos arquivos

In [ ]:
!pip install openpyxl pypdf -q

In [ ]:
import glob
import re
import pandas as pd
from openpyxl import load_workbook

pd.set_option("display.max_colwidth", 120)

# Faz upload manual se estiver rodando no Colab; se não, assume que os arquivos
# já estão na pasta atual (por exemplo, se você montou o Google Drive).
try:
    from google.colab import files
    print("Ambiente Colab detectado. Selecione os 3 arquivos originais no diálogo abaixo:")
    uploaded = files.upload()
except ImportError:
    print("Ambiente não é o Colab: assumindo que os arquivos já estão no diretório atual.")

def encontrar_arquivo(padrao):
    candidatos = glob.glob(padrao)
    if not candidatos:
        raise FileNotFoundError(
            f"Não encontrei nenhum arquivo correspondente a '{padrao}'. "
            "Confirme se o upload foi feito e se o nome do arquivo não mudou."
        )
    if len(candidatos) > 1:
        print(f"Aviso: mais de um arquivo bate com '{padrao}': {candidatos}. Usando o primeiro.")
    return candidatos[0]

CAMINHO_FORMULARIOS = encontrar_arquivo("*Formularios_Tecnicos*.xlsx")
CAMINHO_INDICADORES = encontrar_arquivo("*Indicadores_AP2024*.xlsx")
CAMINHO_GLOSSARIO = encontrar_arquivo("*GLOSSARIO*SINISA*.pdf")

print("Arquivo 1 (formulários técnicos):", CAMINHO_FORMULARIOS)
print("Arquivo 2 (indicadores):", CAMINHO_INDICADORES)
print("Glossário (PDF):", CAMINHO_GLOSSARIO)

## 1. Funções auxiliares de diagnóstico

Essas planilhas do SINISA têm um cabeçalho "empilhado": título do ministério,
título do sistema, título do módulo, data de geração, uma linha em branco, e só
então o cabeçalho real de colunas — seguido por linhas de fórmula/unidade/código
antes dos dados começarem de fato. Em vez de assumir um número fixo de linhas,
detectamos isso procurando marcadores conhecidos.

In [ ]:
def carregar_aba_bruta(caminho_arquivo, nome_aba):
    """Carrega uma aba inteira sem cabeçalho, célula por célula, como strings/valores brutos."""
    wb = load_workbook(caminho_arquivo, read_only=True, data_only=True)
    ws = wb[nome_aba]
    dados = list(ws.iter_rows(values_only=True))
    return pd.DataFrame(dados)

def localizar_linha_cabecalho(df_bruto, marcador="Código IBGE", coluna=0, limite_busca=20):
    """Retorna o índice (0-based) da linha onde o cabeçalho real de colunas começa."""
    for i in range(min(limite_busca, len(df_bruto))):
        valor = df_bruto.iloc[i, coluna]
        if isinstance(valor, str) and valor.strip() == marcador:
            return i
    raise ValueError(f"Não encontrei a linha de cabeçalho (marcador '{marcador}' na coluna {coluna}).")

def localizar_inicio_dos_dados(df_bruto, coluna=0, a_partir_de=0):
    """Retorna o índice (0-based) da primeira linha cujo valor na coluna indicada
    parece um código IBGE de município (inteiro de 7 dígitos)."""
    for i in range(a_partir_de, len(df_bruto)):
        valor = df_bruto.iloc[i, coluna]
        if isinstance(valor, (int, float)) and not pd.isna(valor):
            if len(str(int(valor))) == 7:
                return i
    raise ValueError("Não encontrei o início dos dados (nenhum código IBGE de 7 dígitos encontrado).")

def montar_amostra(df_bruto, linha_cabecalho, linha_inicio_dados, n_linhas=5):
    """Aplica o cabeçalho encontrado e devolve as n_linhas primeiras linhas de dados reais."""
    nomes_colunas = df_bruto.iloc[linha_cabecalho].tolist()
    amostra = df_bruto.iloc[linha_inicio_dados: linha_inicio_dados + n_linhas].copy()
    amostra.columns = nomes_colunas
    return amostra.reset_index(drop=True)

## 2. Arquivo 1 — Formulários Técnicos

### 2.1 Abas disponíveis

In [ ]:
wb1 = load_workbook(CAMINHO_FORMULARIOS, read_only=True)
print("Abas encontradas no arquivo 1:", wb1.sheetnames)

### 2.2 Localizando as informações-alvo

Pelo nome das abas, a aba **"Dados sobre Gestão de Riscos"** é a candidata a
conter tanto "existe mapeamento de área de risco" quanto "quantidade de
domicílios em risco de inundação" — vamos confirmar isso vasculhando os nomes
reais de coluna, não assumindo pelo nome da aba.

In [ ]:
ABA_RISCOS = "Dados sobre Gestão de Riscos"

bruto_riscos = carregar_aba_bruta(CAMINHO_FORMULARIOS, ABA_RISCOS)
linha_cab_riscos = localizar_linha_cabecalho(bruto_riscos)
linha_dados_riscos = localizar_inicio_dos_dados(bruto_riscos, a_partir_de=linha_cab_riscos)

print(f"Aba '{ABA_RISCOS}':")
print(f"  -> Cabeçalho de colunas na linha {linha_cab_riscos} (0-based) = linha {linha_cab_riscos + 1} do Excel")
print(f"  -> Dados começam na linha {linha_dados_riscos} (0-based) = linha {linha_dados_riscos + 1} do Excel")
print(f"  -> Total de linhas na aba: {len(bruto_riscos)}")

In [ ]:
nomes_colunas_riscos = bruto_riscos.iloc[linha_cab_riscos].tolist()

# Busca por substring nos nomes de coluna reais - sem assumir o texto exato de antemão
def buscar_colunas(nomes_colunas, *termos):
    encontrados = []
    for idx, nome in enumerate(nomes_colunas):
        if isinstance(nome, str) and all(t.lower() in nome.lower() for t in termos):
            encontrados.append((idx, nome))
    return encontrados

candidatas_mapeamento = buscar_colunas(nomes_colunas_riscos, "mapeamento", "risco")
candidatas_quantidade = buscar_colunas(nomes_colunas_riscos, "quantidade", "domic", "risco", "inunda")
candidatas_critico = buscar_colunas(nomes_colunas_riscos, "crítico")

print("Colunas candidatas para 'existe mapeamento de área de risco':")
for idx, nome in candidatas_mapeamento:
    print(f"  col {idx}: {nome!r}")

print("\nColunas candidatas para 'quantidade de domicílios em risco de inundação':")
for idx, nome in candidatas_quantidade:
    print(f"  col {idx}: {nome!r}")

print("\nColunas candidatas para 'município crítico':")
for idx, nome in candidatas_critico:
    print(f"  col {idx}: {nome!r}")

Também precisamos do código de campo do SINISA (ex.: `GAP2105`) de cada coluna
candidata, para depois cruzar com o glossário em PDF. Os códigos ficam numa
linha própria, entre o cabeçalho e o início dos dados.

In [ ]:
# A linha de códigos fica a 3 linhas do cabeçalho, mas confirmamos procurando
# uma linha que contenha strings no padrão GAPxxxx / GAPxxxx* entre o cabeçalho e os dados
def localizar_linha_de_codigos(df_bruto, linha_cabecalho, linha_dados):
    padrao_codigo = re.compile(r"^[A-Z]{2,4}\d{3,4}\*?$")
    for i in range(linha_cabecalho + 1, linha_dados):
        valores = df_bruto.iloc[i].tolist()
        acertos = sum(1 for v in valores if isinstance(v, str) and padrao_codigo.match(v.strip()))
        if acertos >= 3:
            return i
    return None

linha_codigos_riscos = localizar_linha_de_codigos(bruto_riscos, linha_cab_riscos, linha_dados_riscos)
print(f"Linha de códigos de campo detectada: índice {linha_codigos_riscos} (0-based) = linha {linha_codigos_riscos + 1} do Excel\n")

codigos_riscos = bruto_riscos.iloc[linha_codigos_riscos].tolist() if linha_codigos_riscos is not None else None

for etiqueta, candidatas in [
    ("mapeamento de área de risco", candidatas_mapeamento),
    ("quantidade de domicílios em risco", candidatas_quantidade),
    ("crítico", candidatas_critico),
]:
    print(f"--- {etiqueta} ---")
    for idx, nome in candidatas:
        codigo = codigos_riscos[idx] if codigos_riscos is not None else "?"
        print(f"  col {idx} | código: {codigo!r} | nome completo: {nome!r}")
    print()

### 2.3 Amostra de 5 linhas — Arquivo 1 (aba "Dados sobre Gestão de Riscos")

Mostrando as colunas de identificação do município + as colunas-alvo encontradas acima.

In [ ]:
colunas_interesse_riscos = [0, 1, 2, 5] + [idx for idx, _ in candidatas_mapeamento] + [idx for idx, _ in candidatas_quantidade]
colunas_interesse_riscos = sorted(set(colunas_interesse_riscos))

amostra_riscos = montar_amostra(bruto_riscos, linha_cab_riscos, linha_dados_riscos, n_linhas=5)
amostra_riscos.iloc[:, colunas_interesse_riscos]

### 2.4 Cruzando com o glossário oficial (PDF)

Extraindo o texto do PDF e procurando pela definição oficial de cada código
encontrado, para confirmar que a coluna significa o que o nome sugere — sem
assumir nada só pelo nome da coluna na planilha.

In [ ]:
from pypdf import PdfReader

leitor = PdfReader(CAMINHO_GLOSSARIO)
texto_glossario = "\n".join(pagina.extract_text() or "" for pagina in leitor.pages)

def mostrar_definicao(codigo, texto_glossario, n_linhas=6):
    codigo_base = codigo.strip("*")
    pos = texto_glossario.find(codigo_base)
    if pos == -1:
        print(f"[{codigo_base}] NÃO encontrado no glossário — não assuma o significado, confirme manualmente.")
        return
    trecho = texto_glossario[pos: pos + 500]
    linhas = [l for l in trecho.split("\n") if l.strip()][:n_linhas]
    print(f"[{codigo_base}] " + " ".join(linhas))
    print()

codigos_para_checar = set()
for _, candidatas in [(None, candidatas_mapeamento), (None, candidatas_quantidade), (None, candidatas_critico)]:
    for idx, _ in candidatas:
        if codigos_riscos is not None and isinstance(codigos_riscos[idx], str):
            codigos_para_checar.add(codigos_riscos[idx])

for codigo in sorted(codigos_para_checar):
    mostrar_definicao(codigo, texto_glossario)

## 3. Arquivo 2 — Indicadores AP2024

### 3.1 Abas disponíveis

In [ ]:
wb2 = load_workbook(CAMINHO_INDICADORES, read_only=True)
print("Abas encontradas no arquivo 2:", wb2.sheetnames)

In [ ]:
ABA_INDICADORES = wb2.sheetnames[0]

bruto_indicadores = carregar_aba_bruta(CAMINHO_INDICADORES, ABA_INDICADORES)
linha_cab_ind = localizar_linha_cabecalho(bruto_indicadores)
linha_dados_ind = localizar_inicio_dos_dados(bruto_indicadores, a_partir_de=linha_cab_ind)

print(f"Aba '{ABA_INDICADORES}':")
print(f"  -> Cabeçalho de colunas na linha {linha_cab_ind} (0-based) = linha {linha_cab_ind + 1} do Excel")
print(f"  -> Dados começam na linha {linha_dados_ind} (0-based) = linha {linha_dados_ind + 1} do Excel")
print(f"  -> Total de linhas na aba: {len(bruto_indicadores)}")

### 3.2 Localizando as informações-alvo

Este arquivo já traz indicadores calculados pelo governo — cada coluna tem,
logo abaixo do nome, a fórmula usada (em termos dos códigos brutos do
formulário técnico) e o código do próprio indicador (ex.: `IGR0001`). Isso já
serve como validação própria, sem precisar do glossário do arquivo 1.

In [ ]:
nomes_colunas_ind = bruto_indicadores.iloc[linha_cab_ind].tolist()

candidatas_critico_ind = buscar_colunas(nomes_colunas_ind, "crítico")
candidatas_drenagem_vias = buscar_colunas(nomes_colunas_ind, "vias", "rede")
candidatas_investimento = buscar_colunas(nomes_colunas_ind, "investimento", "drenagem")
candidatas_domicilios_risco_pct = buscar_colunas(nomes_colunas_ind, "domic", "risco", "inunda")

# As 2 linhas seguintes ao cabeçalho trazem fórmula e unidade; o código do indicador
# vem 3 linhas abaixo do cabeçalho (mesma lógica de sanity-check usada no arquivo 1)
linha_formula_ind = linha_cab_ind + 1
linha_unidade_ind = linha_cab_ind + 2
linha_codigo_ind = linha_cab_ind + 3

formulas_ind = bruto_indicadores.iloc[linha_formula_ind].tolist()
unidades_ind = bruto_indicadores.iloc[linha_unidade_ind].tolist()
codigos_ind = bruto_indicadores.iloc[linha_codigo_ind].tolist()

# valida que a linha de código realmente parece códigos de indicador (ex.: IGR0001)
padrao_codigo_indicador = re.compile(r"^[A-Z]{2,4}\d{3,4}\*?$")
amostra_valores = [v for v in codigos_ind if isinstance(v, str)]
proporcao_valida = sum(1 for v in amostra_valores if padrao_codigo_indicador.match(v.strip())) / max(len(amostra_valores), 1)
print(f"Linha {linha_codigo_ind} parece ser a linha de códigos de indicador? "
      f"{proporcao_valida:.0%} dos valores batem com o padrão esperado.\n")

for etiqueta, candidatas in [
    ("município crítico", candidatas_critico_ind),
    ("% vias com rede de drenagem", candidatas_drenagem_vias),
    ("investimento per capita em drenagem", candidatas_investimento),
    ("% domicílios em risco de inundação", candidatas_domicilios_risco_pct),
]:
    print(f"--- {etiqueta} ---")
    for idx, nome in candidatas:
        print(f"  col {idx} | código: {codigos_ind[idx]!r} | fórmula: {formulas_ind[idx]!r} | unidade: {unidades_ind[idx]!r}")
        print(f"           nome completo: {nome!r}")
    print()

### 3.3 Amostra de 5 linhas — Arquivo 2 (aba de indicadores)

In [ ]:
colunas_interesse_ind = [0, 1, 2]
for candidatas in [candidatas_critico_ind, candidatas_drenagem_vias, candidatas_investimento, candidatas_domicilios_risco_pct]:
    colunas_interesse_ind += [idx for idx, _ in candidatas]
colunas_interesse_ind = sorted(set(colunas_interesse_ind))

amostra_ind = montar_amostra(bruto_indicadores, linha_cab_ind, linha_dados_ind, n_linhas=5)
amostra_ind.iloc[:, colunas_interesse_ind]

## 4. Resumo do diagnóstico (colunas confirmadas)

| Informação | Arquivo | Aba | Linha do cabeçalho (Excel) | Linha de início dos dados (Excel) | Coluna encontrada |
|---|---|---|---|---|---|
| Domicílios em risco de inundação (quantidade) — **GAP2110**, não GAP2109 | 1 | Dados sobre Gestão de Riscos | ver `linha_cab_riscos+1` acima | ver `linha_dados_riscos+1` acima | ver `candidatas_quantidade` |
| Existe mapeamento de área de risco — **GAP2105** | 1 | Dados sobre Gestão de Riscos | idem | idem | ver `candidatas_mapeamento` |
| Município crítico | 2 | Indicadores por município | ver `linha_cab_ind+1` acima | ver `linha_dados_ind+1` acima | ver `candidatas_critico_ind` |
| % vias com rede de drenagem — **IAP0002** | 2 | Indicadores por município | idem | idem | ver `candidatas_drenagem_vias` |
| Investimento per capita em drenagem — **IFD0010** | 2 | Indicadores por município | idem | idem | ver `candidatas_investimento` |
| % domicílios em risco (indicador pronto) — **IGR0001**, mantido junto com o valor absoluto do GAP2110 | 2 | Indicadores por município | idem | idem | ver `candidatas_domicilios_risco_pct` |

**Colunas validadas.** Cada uma foi selecionada por código exato (ou, no caso
de `Crítico`, por conferência da definição de GAP2001 no glossário) — não por
posição ou nome aproximado. O notebook segue abaixo para a limpeza
estrutural, a padronização do `Cod_IBGE_7` e a exportação do `sinisa_limpo.csv`.

## 5. Etapa 2 — Limpeza estrutural

Aqui isolamos so as colunas confirmadas, padronizamos o codigo do municipio e
juntamos os dois arquivos. Nenhuma regra de negocio e decidida nesta etapa —
os casos de domicilios "0 em risco" e os valores vazios sao apenas contados e
documentados, nao tratados.

In [ ]:
log_transformacoes = []

def registrar_passo(nome, linhas_antes, linhas_depois, motivo):
    log_transformacoes.append({
        "Etapa": nome,
        "Linhas antes": linhas_antes,
        "Linhas depois": linhas_depois,
        "Motivo": motivo,
    })
    print(f"[{nome}] {linhas_antes} -> {linhas_depois} linhas. Motivo: {motivo}")

### 5.1 Selecionando as colunas confirmadas por codigo (nao por posicao)

Em vez de usar indice de coluna fixo (fragil se alguem reordenar colunas na
planilha), selecionamos cada coluna pelo codigo oficial do SINISA que ja
identificamos e voce confirmou.

In [ ]:
def selecionar_por_codigo(candidatas, codigos, codigo_alvo):
    """Dentre as colunas candidatas (lista de (idx, nome)), retorna a que tem
    o codigo exato pedido. Levanta erro claro se nao achar - preferimos falhar
    a adivinhar."""
    for idx, nome in candidatas:
        codigo_col = codigos[idx]
        if isinstance(codigo_col, str) and codigo_col.strip().rstrip("*") == codigo_alvo:
            return idx, nome
    raise ValueError(
        f"Nao encontrei nenhuma coluna candidata com o codigo '{codigo_alvo}'. "
        f"Candidatas disponiveis: {[(i, codigos[i]) for i, _ in candidatas]}"
    )

# --- Arquivo 1: mapeamento (GAP2105) e quantidade de domicilios em risco (GAP2110) ---
idx_mapeamento, nome_mapeamento = selecionar_por_codigo(candidatas_mapeamento, codigos_riscos, "GAP2105")
idx_qtd_domicilios, nome_qtd_domicilios = selecionar_por_codigo(candidatas_quantidade, codigos_riscos, "GAP2110")

print(f"Mapeamento de risco -> col {idx_mapeamento}: {nome_mapeamento!r} (GAP2105)")
print(f"Quantidade de domicilios em risco -> col {idx_qtd_domicilios}: {nome_qtd_domicilios!r} (GAP2110)")

# --- Arquivo 2: critico, % vias com drenagem (IAP0002), investimento per capita (IFD0010), % domicilios risco (IGR0001) ---
def selecionar_critico_validado_por_glossario(candidatas, texto_glossario, codigo_glossario="GAP2001"):
    """A coluna 'Critico' do arquivo 2 nao carrega um codigo de campo proprio na
    linha de codigos do indicador (fica '-' ali) - o codigo GAP2001 que a
    identifica vem do glossario do arquivo 1. Por isso a validacao aqui e por
    unicidade (erro se houver mais de 1 candidata) + checagem de que a definicao
    de GAP2001 no glossario realmente fala de municipio critico, em vez de so
    pegar a primeira candidata silenciosamente."""
    if len(candidatas) != 1:
        raise ValueError(
            f"Esperava exatamente 1 coluna candidata para 'critico', encontrei {len(candidatas)}: "
            f"{candidatas}. Ajuste a selecao antes de prosseguir - nao escolha a primeira silenciosamente."
        )
    idx, nome = candidatas[0]
    pos = texto_glossario.find(codigo_glossario)
    if pos == -1:
        raise ValueError(f"Codigo {codigo_glossario} nao encontrado no glossario - nao consigo validar a coluna 'Critico'.")
    trecho = texto_glossario[pos: pos + 300].lower()
    if "crítico" not in trecho and "critico" not in trecho:
        raise ValueError(
            f"A definicao de {codigo_glossario} no glossario nao menciona 'critico' - "
            "a coluna pode nao ser a esperada."
        )
    return idx, nome

idx_critico, nome_critico = selecionar_critico_validado_por_glossario(candidatas_critico_ind, texto_glossario)
idx_pct_vias, nome_pct_vias = selecionar_por_codigo(candidatas_drenagem_vias, codigos_ind, "IAP0002")
idx_investimento, nome_investimento = selecionar_por_codigo(candidatas_investimento, codigos_ind, "IFD0010")
idx_pct_domicilios, nome_pct_domicilios = selecionar_por_codigo(candidatas_domicilios_risco_pct, codigos_ind, "IGR0001")

print(f"\nCritico -> col {idx_critico}: {nome_critico!r}")
print(f"% vias com rede de drenagem -> col {idx_pct_vias}: {nome_pct_vias!r} (IAP0002)")
print(f"Investimento per capita em drenagem -> col {idx_investimento}: {nome_investimento!r} (IFD0010)")
print(f"% domicilios em risco (indicador pronto) -> col {idx_pct_domicilios}: {nome_pct_domicilios!r} (IGR0001)")

### 5.2 Extraindo apenas as colunas relevantes de cada arquivo

In [ ]:
# Todas as linhas de dados de cada aba (sem cabecalho ainda)
dados_riscos_completo = bruto_riscos.iloc[linha_dados_riscos:].reset_index(drop=True)
dados_ind_completo = bruto_indicadores.iloc[linha_dados_ind:].reset_index(drop=True)

registrar_passo(
    "Carregar dados brutos (arquivo 1, aba Gestao de Riscos)",
    linhas_antes=len(bruto_riscos),
    linhas_depois=len(dados_riscos_completo),
    motivo="Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados."
)
registrar_passo(
    "Carregar dados brutos (arquivo 2, Indicadores)",
    linhas_antes=len(bruto_indicadores),
    linhas_depois=len(dados_ind_completo),
    motivo="Descartadas as linhas de titulo/cabecalho/formula/unidade/codigo acima do inicio real dos dados."
)

df1 = dados_riscos_completo.iloc[:, [0, 1, 2, idx_mapeamento, idx_qtd_domicilios]].copy()
df1.columns = ["Cod_IBGE_7", "Nome", "UF", "mapeamento_area_risco", "qtd_domicilios_risco"]

df2 = dados_ind_completo.iloc[:, [0, 1, 2, idx_critico, idx_pct_vias, idx_investimento, idx_pct_domicilios]].copy()
df2.columns = ["Cod_IBGE_7", "Nome", "UF", "municipio_critico", "pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"]

print("df1 (arquivo 1, colunas isoladas):", df1.shape)
display(df1.head())
print("\ndf2 (arquivo 2, colunas isoladas):", df2.shape)
display(df2.head())

### 5.3 Padronizando `Cod_IBGE_7` como inteiro de 7 digitos nos dois arquivos

In [ ]:
for nome_df, df in [("df1", df1), ("df2", df2)]:
    antes = len(df)
    cod_numerico = pd.to_numeric(df["Cod_IBGE_7"], errors="coerce")
    invalidos = cod_numerico.isna().sum()
    df["Cod_IBGE_7"] = cod_numerico

    tamanhos = df["Cod_IBGE_7"].dropna().astype("int64").astype(str).str.len()
    fora_do_padrao = (tamanhos != 7).sum()

    print(f"{nome_df}: {invalidos} valor(es) de Cod_IBGE_7 nao numerico(s); "
          f"{fora_do_padrao} valor(es) numerico(s) mas fora do padrao de 7 digitos.")

In [ ]:
# Converte para Int64 nullable (permite manter NaN sem virar float com casas decimais)
df1["Cod_IBGE_7"] = df1["Cod_IBGE_7"].astype("Int64")
df2["Cod_IBGE_7"] = df2["Cod_IBGE_7"].astype("Int64")

print("dtype Cod_IBGE_7 em df1:", df1["Cod_IBGE_7"].dtype)
print("dtype Cod_IBGE_7 em df2:", df2["Cod_IBGE_7"].dtype)

### 5.4 Removendo linhas sem codigo de municipio (rodapes, totais, linhas em branco)

In [ ]:
antes_df1 = len(df1)
df1 = df1.dropna(subset=["Cod_IBGE_7"]).reset_index(drop=True)
registrar_passo(
    "dropna em Cod_IBGE_7 (arquivo 1)",
    linhas_antes=antes_df1,
    linhas_depois=len(df1),
    motivo="Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco)."
)

antes_df2 = len(df2)
df2 = df2.dropna(subset=["Cod_IBGE_7"]).reset_index(drop=True)
registrar_passo(
    "dropna em Cod_IBGE_7 (arquivo 2)",
    linhas_antes=antes_df2,
    linhas_depois=len(df2),
    motivo="Remove linhas sem codigo de municipio (rodapes, totais, linhas em branco)."
)

### 5.5 Tratando a string literal "null" nas colunas categoricas

Ao inspecionar o Excel bruto, os 725 municipios que nao participaram da
pesquisa (os mesmos que ficam sem valor em `qtd_domicilios_risco`) tem, na
coluna de mapeamento de risco, a **string literal `"null"`** em vez de uma
celula realmente vazia - por isso `.isna()` nao pegava esses casos antes.
Aqui convertemos essa string (e variacoes de maiusculas/minusculas e espacos
em volta) para um valor ausente de verdade, em `mapeamento_area_risco` e,
defensivamente, tambem em `municipio_critico` (mesmo que hoje ela nao
apresente o problema).

In [ ]:
def relatar_valores(df, coluna, nome_df, valores_esperados=("Sim", "Não")):
    valores_esperados = ("Sim", "Não")
    contagem = df[coluna].value_counts(dropna=False)
    print(f"--- {nome_df}.{coluna} ---")
    print(contagem)
    inesperados = [v for v in contagem.index if pd.notna(v) and v not in ("Sim", "Não")]
    if inesperados:
        print(f"  Valor(es) fora do padrao esperado (Sim/Não): {inesperados}")
    else:
        print(f"  Nenhum valor fora do padrao esperado (Sim/Não), alem de nulo.")
    print()
    return contagem

def limpar_null_literal(serie):
    def normalizar(v):
        if isinstance(v, str) and v.strip().lower() == "null":
            return pd.NA
        return v
    return serie.apply(normalizar)

print("### ANTES da limpeza ###\n")
relatar_valores(df1, "mapeamento_area_risco", "df1")
relatar_valores(df2, "municipio_critico", "df2")

for nome_df, df, coluna in [("df1", df1, "mapeamento_area_risco"), ("df2", df2, "municipio_critico")]:
    qtd_null_literal = df[coluna].apply(lambda v: isinstance(v, str) and v.strip().lower() == "null").sum()
    df[coluna] = limpar_null_literal(df[coluna])
    print(f"{nome_df}.{coluna}: {qtd_null_literal} string(s) 'null' literal convertida(s) para nulo de verdade.")

print("\n### DEPOIS da limpeza ###\n")
relatar_valores(df1, "mapeamento_area_risco", "df1")
relatar_valores(df2, "municipio_critico", "df2")

### 5.6 Convertendo as colunas numericas e contando quantas linhas foram afetadas

`municipio_critico` e `mapeamento_area_risco` sao categoricas (Sim/Nao) e
ficam como estao — so as colunas realmente numericas passam por
`pd.to_numeric(errors='coerce')`.

In [ ]:
colunas_numericas_df1 = ["qtd_domicilios_risco"]
colunas_numericas_df2 = ["pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"]

def converter_numericas(df, colunas, nome_df):
    for col in colunas:
        valores_antes_nulos = df[col].isna().sum()
        convertido = pd.to_numeric(df[col], errors="coerce")
        novos_nulos = convertido.isna().sum() - valores_antes_nulos
        df[col] = convertido
        print(f"{nome_df}.{col}: {novos_nulos} linha(s) tinham valor nao numerico "
              f"(ex.: '-', texto, espaco em branco) e viraram NaN na conversao.")

converter_numericas(df1, colunas_numericas_df1, "df1")
converter_numericas(df2, colunas_numericas_df2, "df2")

### 5.7 Percentual de preenchimento de cada coluna extraida

In [ ]:
def percentual_preenchimento(df, colunas, nome_df):
    linhas = []
    for col in colunas:
        preenchido = df[col].notna().mean() * 100
        linhas.append({"Arquivo": nome_df, "Coluna": col, "% preenchido": round(preenchido, 1)})
    return linhas

resumo_preenchimento = (
    percentual_preenchimento(df1, ["mapeamento_area_risco", "qtd_domicilios_risco"], "df1")
    + percentual_preenchimento(df2, ["municipio_critico", "pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"], "df2")
)

df_preenchimento = pd.DataFrame(resumo_preenchimento)
display(df_preenchimento)

### 5.8 Documentando (sem decidir) os casos de "0 domicilios em risco" e valores vazios

Isso e regra de negocio ainda nao fechada pela squad — aqui so contamos, nao
tratamos.

In [ ]:
qtd_zero = (df1["qtd_domicilios_risco"] == 0).sum()
qtd_vazio_df1 = df1["qtd_domicilios_risco"].isna().sum()
qtd_vazio_mapeamento = df1["mapeamento_area_risco"].isna().sum()

print(f"df1: {qtd_zero} municipio(s) declaram 0 domicilios em risco de inundacao (GAP2110).")
print(f"df1: {qtd_vazio_df1} municipio(s) sem valor para domicilios em risco (vazio).")
print(f"df1: {qtd_vazio_mapeamento} municipio(s) sem resposta para 'existe mapeamento de area de risco'.")

for col in ["municipio_critico", "pct_vias_rede_drenagem", "investimento_percapita_drenagem", "pct_domicilios_risco"]:
    vazios = df2[col].isna().sum()
    print(f"df2.{col}: {vazios} valor(es) vazio(s).")

print("\nNenhuma decisao foi tomada sobre esses casos (nao foram removidos, zerados ou imputados) - "
      "fica para a squad decidir a regra de negocio.")

### 5.9 Juncao dos dois arquivos por `Cod_IBGE_7`

Juncao externa (`outer`): mantem municipios mesmo quando so aparecem em um dos
dois arquivos.

In [ ]:
codigos_df1 = set(df1["Cod_IBGE_7"])
codigos_df2 = set(df2["Cod_IBGE_7"])

so_df1 = codigos_df1 - codigos_df2
so_df2 = codigos_df2 - codigos_df1
em_ambos = codigos_df1 & codigos_df2

print(f"Municipios so no arquivo 1: {len(so_df1)}")
print(f"Municipios so no arquivo 2: {len(so_df2)}")
print(f"Municipios em ambos: {len(em_ambos)}")

df_final = pd.merge(
    df1,
    df2.drop(columns=["Nome", "UF"]),  # ja vem de df1, evita colunas duplicadas
    on="Cod_IBGE_7",
    how="outer",
    validate="one_to_one",
)

registrar_passo(
    "Merge df1 + df2 por Cod_IBGE_7 (outer)",
    linhas_antes=f"{len(df1)} (df1) + {len(df2)} (df2)",
    linhas_depois=len(df_final),
    motivo="Juncao externa para nao perder municipio que so apareceu em um dos dois arquivos."
)

print(f"\nLinhas no dataframe final: {len(df_final)}")
display(df_final.head())

## 6. Etapa 3 — Documentacao obrigatoria (protocolo DataConnect)

### 6.a Log de transformacoes

Gerado dinamicamente a partir de cada passo executado acima (numero de linhas
antes/depois e o motivo de cada decisao).

In [ ]:
df_log = pd.DataFrame(log_transformacoes)
display(df_log)

In [ ]:
from IPython.display import display, Markdown

linhas_md = ["| Etapa | Linhas antes | Linhas depois | Motivo |", "|---|---|---|---|"]
for passo in log_transformacoes:
    linhas_md.append(f"| {passo['Etapa']} | {passo['Linhas antes']} | {passo['Linhas depois']} | {passo['Motivo']} |")

display(Markdown("\n".join(linhas_md)))

## 7. Etapa 4 — Exportacao

In [ ]:
CAMINHO_SAIDA = "sinisa_limpo.csv"
df_final.to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8-sig")

print(f"Exportado: {CAMINHO_SAIDA}")
print(f"Linhas: {len(df_final)} | Colunas: {list(df_final.columns)}")

try:
    from google.colab import files as colab_files
    colab_files.download(CAMINHO_SAIDA)
except ImportError:
    print("Fora do Colab: o arquivo foi salvo no diretorio atual do notebook.")

---

# 2. Tratamento e validação CAPAG

**Notebook de origem:** `Notebook/Tratamento_CAPAG.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

# Tratamento e Validação da CAPAG — Ano-Base 2025
### Posição setembro/2026

**Objetivo:** tratar e validar a base CAPAG utilizada no projeto
Observatório do Clima, preparando uma tabela municipal para integração
com SINISA e Atlas Digital de Desastres.

A base utilizada é a posição de setembro/2026, referente ao ano-base 2025.

## Etapas

1. Carregamento da base oficial utilizada pelo projeto.
2. Validação da versão da CAPAG.
3. Seleção das colunas necessárias.
4. Verificação de nulos e duplicidades.
5. Validação do código IBGE.
6. Padronização dos campos.
7. Criação dos critérios fiscais.
8. Validação final.
9. Exportação do CSV tratado.

## Regra de negócio

A CAPAG é um dos critérios utilizados no funil de priorização.

- C ou D;
- n.d. ou n.e.

Essas categorias formam o grupo utilizado pela regra fiscal de
priorização do projeto.

A integração com SINISA e Atlas será realizada posteriormente pelo
código IBGE de 7 dígitos.


## 1. Importações e configurações



In [ ]:
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

# ---------------------------------------------------------
# CONFIGURAÇÕES DA FONTE
# ---------------------------------------------------------

# CAPAG posição setembro/2026
ARQUIVO_ID = "1ctfcr6QOz5XrQwPVrjl5h_Px52J-aj2-"

URL_EXPORT = (
    f"https://docs.google.com/spreadsheets/d/"
    f"{ARQUIVO_ID}/export?format=xlsx"
)

ABA_CAPAG = "CAPAG Ano Base 2025"

# Contagens de controle da posição setembro/2026
CONTAGENS_ESPERADAS = {
    "A+": 418,
    "A": 1096,
    "B+": 200,
    "B": 963,
    "C": 2274,
    "D": 20,
    "n.d.": 426,
    "n.e.": 170
}

TOTAL_ESPERADO = sum(CONTAGENS_ESPERADAS.values())

print("Total esperado de municípios:", TOTAL_ESPERADO)


# 2. Carregamento da base CAPAG

Leitura da posição de setembro/2026, ano-base 2025.
O código IBGE é carregado como texto porque é um identificador.

In [ ]:
df = pd.read_excel(
    URL_EXPORT,
    sheet_name=ABA_CAPAG,
    header=3,
    dtype={
        "Código Município Completo": "string"
    }
)

print("Dimensão da base:", df.shape)

df.head()

## 3. Validar se realmente é setembro

Esta etapa funciona como controle de versão.
As contagens devem corresponder à posição de setembro/2026.

In [ ]:
# Padroniza apenas espaços da classificação
df["CAPAG"] = (
    df["CAPAG"]
    .astype("string")
    .str.strip()
)

distribuicao_inicial = (
    df["CAPAG"]
    .value_counts(dropna=False)
)

print(distribuicao_inicial)

In [ ]:
# Validação automática das contagens esperadas
for nota, quantidade_esperada in CONTAGENS_ESPERADAS.items():

    quantidade_encontrada = int(
        distribuicao_inicial.get(nota, 0)
    )

    assert quantidade_encontrada == quantidade_esperada, (
        f"ERRO na nota {nota}: "
        f"esperado {quantidade_esperada}, "
        f"encontrado {quantidade_encontrada}"
    )

assert len(df) == TOTAL_ESPERADO, (
    f"Quantidade de municípios inesperada: {len(df)}"
)

print("\n✅ Base CAPAG setembro/2026 validada.")

## 4. Seleção das colunas



In [ ]:
# Seleção das colunas necessárias
colunas_capag = [
    "Código Município Completo",
    "Nome_Município",
    "UF",
    "CAPAG antes do ranking",
    "Ranking da CCONF",
    "CAPAG"
]

# Confere se todas as colunas existem antes de selecionar
colunas_faltantes = [
    coluna
    for coluna in colunas_capag
    if coluna not in df.columns
]

assert not colunas_faltantes, (
    f"Colunas não encontradas: {colunas_faltantes}"
)

df_capag = df[colunas_capag].copy()

print("Dimensão da base selecionada:", df_capag.shape)

df_capag.head()

## 5. Qualidade dos dados

Verificação de valores ausentes, linhas duplicadas e duplicidade
do código IBGE.

In [ ]:
# Validação de nulos e duplicidades
print("Valores nulos por coluna:")
print(df_capag.isna().sum())

print("\nLinhas totalmente duplicadas:")
duplicadas = df_capag.duplicated().sum()
print(duplicadas)

print("\nCódigos IBGE duplicados:")
duplicados_ibge = (
    df_capag["Código Município Completo"]
    .duplicated()
    .sum()
)
print(duplicados_ibge)

# Campos indispensáveis
campos_obrigatorios = [
    "Código Município Completo",
    "Nome_Município",
    "UF",
    "CAPAG"
]

assert df_capag[campos_obrigatorios].isna().sum().sum() == 0, (
    "Existem valores nulos em campos obrigatórios."
)

assert duplicadas == 0, (
    "Existem linhas duplicadas."
)

assert duplicados_ibge == 0, (
    "Existem códigos IBGE duplicados."
)

print("\n✅ Nulos e duplicidades validados.")

## 6. Código IBGE

O código IBGE é tratado como identificador textual e deve possuir
exatamente 7 dígitos.

In [ ]:
# Validação do código IBGE
df_capag["Código Município Completo"] = (
    df_capag["Código Município Completo"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

codigo_valido = (
    df_capag["Código Município Completo"]
    .str.fullmatch(r"\d{7}", na=False)
)

print(
    "Códigos válidos com 7 dígitos:",
    codigo_valido.sum()
)

print(
    "Códigos inválidos:",
    (~codigo_valido).sum()
)

assert codigo_valido.all(), (
    "Foram encontrados códigos IBGE fora do padrão de 7 dígitos."
)

assert (
    df_capag["Código Município Completo"].nunique()
    == TOTAL_ESPERADO
), "Código IBGE não é único."

print("\n✅ Código IBGE validado.")

## 7. Padronização dos campos


In [ ]:

# Padronização dos campos
df_capag["Nome_Município"] = (
    df_capag["Nome_Município"]
    .astype("string")
    .str.strip()
)

df_capag["UF"] = (
    df_capag["UF"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_capag["CAPAG"] = (
    df_capag["CAPAG"]
    .astype("string")
    .str.strip()
)

df_capag["CAPAG antes do ranking"] = (
    df_capag["CAPAG antes do ranking"]
    .astype("string")
    .str.strip()
)

df_capag["Ranking da CCONF"] = (
    df_capag["Ranking da CCONF"]
    .astype("string")
    .str.strip()
)

print("Padronização concluída.")

## 8. Padronizar nomes das colunas

In [ ]:
df_capag_final = df_capag.rename(
    columns={
        "Código Município Completo": "cod_ibge",
        "Nome_Município": "municipio",
        "UF": "uf",
        "CAPAG antes do ranking": "capag_pre_ranking",
        "Ranking da CCONF": "ranking_cconf",
        "CAPAG": "capag"
    }
).copy()

df_capag_final.head()

## 9. Criar as regras fiscais
São preservados dois indicadores:

- `criterio_capag_cd`: regra anterior do projeto C/D.

- `criterio_fiscal_priorizacao`: inclui C, D, n.d. e n.e.

Também será criada uma flag específica para municípios sem nota.

In [ ]:
# ---------------------------------------------------------
# REGRA ANTIGA / CONTROLE
# ---------------------------------------------------------

df_capag_final["criterio_capag_cd"] = (
    df_capag_final["capag"]
    .isin(["C", "D"])
)


# ---------------------------------------------------------
# MUNICÍPIOS SEM NOTA
# ---------------------------------------------------------

df_capag_final["sem_nota_capag"] = (
    df_capag_final["capag"]
    .isin(["n.d.", "n.e."])
)


# ---------------------------------------------------------
# REGRA FISCAL APROVADA PELA CÉLULA EM 07/10
# ---------------------------------------------------------

categorias_priorizadas = [
    "C",
    "D",
    "n.d.",
    "n.e."
]

df_capag_final["criterio_fiscal_priorizacao"] = (
    df_capag_final["capag"]
    .isin(categorias_priorizadas)
)


print("CAPAG C/D:")
print(
    df_capag_final["criterio_capag_cd"]
    .value_counts()
)

print("\nSem nota CAPAG:")
print(
    df_capag_final["sem_nota_capag"]
    .value_counts()
)

print("\nCritério fiscal de priorização:")
print(
    df_capag_final["criterio_fiscal_priorizacao"]
    .value_counts()
)

## 10. Validação final

In [ ]:
print("Dimensão final:", df_capag_final.shape)

print("\nDistribuição CAPAG final:")
distribuicao_final = (
    df_capag_final["capag"]
    .value_counts(dropna=False)
)

print(distribuicao_final)


# ---------------------------------------------------------
# TESTES AUTOMÁTICOS
# ---------------------------------------------------------

assert len(df_capag_final) == 5567, (
    "Quantidade final de municípios incorreta."
)

assert df_capag_final["cod_ibge"].nunique() == 5567, (
    "Código IBGE não está único."
)

assert df_capag_final["cod_ibge"].duplicated().sum() == 0, (
    "Existem códigos IBGE duplicados."
)

assert (
    df_capag_final["cod_ibge"]
    .str.fullmatch(r"\d{7}", na=False)
    .all()
), "Existem códigos IBGE inválidos."


# Distribuição CAPAG
for nota, quantidade_esperada in CONTAGENS_ESPERADAS.items():

    quantidade_encontrada = int(
        distribuicao_final.get(nota, 0)
    )

    assert quantidade_encontrada == quantidade_esperada, (
        f"Erro na distribuição da nota {nota}: "
        f"{quantidade_encontrada}"
    )


# Critério antigo C/D
assert (
    df_capag_final["criterio_capag_cd"].sum()
    == 2294
), "Quantidade C/D inesperada."


# Municípios sem nota
assert (
    df_capag_final["sem_nota_capag"].sum()
    == 596
), "Quantidade de municípios sem nota inesperada."


# Nova regra fiscal
assert (
    df_capag_final["criterio_fiscal_priorizacao"].sum()
    == 2890
), "Quantidade do critério fiscal inesperada."


print("\n✅ TODAS AS VALIDAÇÕES FORAM APROVADAS.")

## 11. Exportação

In [ ]:
caminho_saida = Path(
    "/content/capag_nacional_tratada.csv"
)

df_capag_final.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Arquivo exportado com sucesso.")
print("Caminho:", caminho_saida)
print("Dimensão:", df_capag_final.shape)

## 12. Conferência do arquivo que realmente foi exportado

In [ ]:
df_exportado = pd.read_csv(
    caminho_saida,
    dtype={
        "cod_ibge": "string"
    },
    encoding="utf-8-sig"
)

print("Dimensão do CSV exportado:", df_exportado.shape)

print("\nDistribuição da CAPAG no CSV:")
print(
    df_exportado["capag"]
    .value_counts(dropna=False)
)

assert len(df_exportado) == 5567

assert df_exportado["cod_ibge"].nunique() == 5567

assert df_exportado["cod_ibge"].duplicated().sum() == 0

print("\n✅ CSV exportado e relido com sucesso.")

---

# 3. Diagnóstico exploratório Atlas/S2iD

**Notebook de origem:** `Notebook/Cópia_de_Exploração_Atlas_S2iD.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.


##Diagnóstico de Qualidade de Dados: Desastres Hidrológicos no S2iD (2021–2025)

**Objetivo Metodológico:** Este notebook realiza uma **auditoria estática e não destrutiva** sobre a base de dados do Atlas Digital de Desastres (S2iD). O propósito aqui **não é apagar ou alterar precipitadamente os dados brutos**, mas sim "tirar uma radiografia" completa do dataset para identificar anomalias, criar etiquetas de auditoria (*flags* booleanas e colunas calculadas) e fundamentar a seção de qualidade da documentação técnica e do nosso **Data Card**.

---

### Fundamentação Teórica e Referência Metodológica

O método de diagnóstico e a avaliação por dimensões aplicadas neste código foram estruturadas com base nos princípios de qualidade de dados consolidados na revisão sistemática da literatura de **Lemos, Coelho Júnior e Martins (2023)**, integrando preceitos clássicos de governança e auditoria de bancos de dados (*Chapman, 2005; Eckerson, 2002; Batini &amp; Scannapieca, 2006*):

&gt; **Referência Bibliográfica de Suporte:**  
&gt; LEMOS, Daniela Lucas da Silva; COELHO JÚNIOR, Abeil; MARTINS, Dalton Lopes. **Modelos de diagnóstico de qualidade de dados no domínio do Patrimônio Cultural: uma revisão sistemática de literatura**. *Perspectivas em Ciência da Informação*, v. 28, e46064, 2023. DOI: [https://doi.org/10.1590/1981-5344/46064](https://doi.org/10.1590/1981-5344/46064).

---

### Escopo e Dimensões de Diagnóstico Avaliadas

Carregamos a base bruta do S2iD, aplicamos o filtro do projeto (**Brasil, 2021 a 2025, Desastres Hidrológicos - totalizando 9.890 ocorrências**) e submetemos o dataset às **6 dimensões essenciais de qualidade**:

1. **Completude:** Avaliação de nulos físicos (sintáticos) e sinalização de "falsos zeros" em prejuízos financeiros (semântica).
2. **Consistência Interna:** Validação da coerência matemática das subcategorias humanas e regras de status legal.
3. **Unicidade e Duplicidade:** Rastreamento de duplicatas por protocolo de sistema e por chave semântica (`IBGE + Data + Tipologia`).
4. **Acurácia e Outliers:** Mapeamento estatístico de valores extremos (\\(3 \times IQR\\)) para isolar catástrofes históricas atípicas de prejuízo típico.
5. **Atualidade e Cobertura Temporal:** Validação da conversão de datas e continuidade da série temporal de 5 anos.
6. **Conformidade de Formato:** Validação por Expressões Regulares (*Regex*) de códigos IBGE (7 dígitos) e siglas de UF.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Fora do Colab: lendo atlas_geral.csv da pasta atual.")

In [ ]:
# Importamos a biblioteca 'pandas', que é a ferramenta padrão do Python para ler e analisar tabelas
import pandas as pd

# 1. Carregamento da base de dados
# Indicamos o nome exato do arquivo. O 'sep=";"' avisa ao computador que as colunas no Brasil são separadas por ponto e vírgula
import os
# Procura primeiro na pasta atual; no Colab, usa o Drive
caminho_arquivo = 'atlas_geral.csv' if os.path.exists('atlas_geral.csv') else '/content/drive/MyDrive/atlas_geral.csv'

print("Lendo o arquivo original...")
df_bruto = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8')

print(f"Base carregada com sucesso! O arquivo inteiro tem {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

## 1. Aplicando o Recorte do Projeto
A base contém o Brasil inteiro desde 1991. Para avaliar a qualidade do dado que *nós* vamos usar, primeiro precisamos isolar apenas o nosso alvo: estados do Sul (RS, SC, PR), anos de 2021 a 2025, e o grupo Hidrológico (que engloba Inundações, Enxurradas, Alagamentos, Chuvas Intensas e Movimento de Massa).

In [ ]:
df_bruto['Data_Evento_dt'] = pd.to_datetime(df_bruto['Data_Evento'], format='%d/%m/%Y', errors='coerce')

# List of all Brazilian state acronyms
estados_brasil = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG',
    'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
]
filtro_estado = df_bruto['Sigla_UF'].isin(estados_brasil)
filtro_ano = (df_bruto['Data_Evento_dt'].dt.year >= 2021) & (df_bruto['Data_Evento_dt'].dt.year <= 2025)
filtro_grupo = df_bruto['grupo_de_desastre'] == 'Hidrológico'

df_recorte = df_bruto[filtro_estado & filtro_ano & filtro_grupo].copy()

print(f"Pronto! Nosso recorte tem exatamente {len(df_recorte)} registros de desastres.")

## 2. Estado Atual da Base (Distribuição Geográfica e Tipologias)
Agora vamos pedir para o computador contar quantas linhas existem para cada estado, quais os tipos exatos de desastres que sobraram no recorte e qual o status de reconhecimento deles no sistema S2iD.

In [ ]:
# O comando 'value_counts()' agrupa os itens iguais e conta quantos existem de cada
print("--- DISTRIBUIÇÃO POR ESTADO ---")
print(df_recorte['Sigla_UF'].value_counts())

print("\n--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---")
print(df_recorte['descricao_tipologia'].value_counts())

print("\n--- STATUS LEGAL DOS DESASTRES ---")
# Mostra se o desastre foi apenas registrado pela prefeitura ou se já foi reconhecido oficialmente pelo Governo
print(df_recorte['Status'].value_counts())


##3. Diagnóstico de Qualidade: Completude (Completeness)

O que foi avaliado: a presença física de dados (nulos) e a validade semântica dos valores preenchidos.

**Completude Sintática (Nulos Físicos)**: o resultado mostrou 0% de campos nulos nas colunas vitais.

**Completude Semântica ("Falsos Zeros")**: o resultado mostrou que 36,1% dos registros (3.567 ocorrências) possuem valor R$ 0,00 simultaneamente em Danos Materiais e Prejuízos Públicos. A interpretação é que isso não indica ausência de estrago real, mas sim o registro de emergência efetuado antes da conclusão do laudo pericial de prejuízos pelo município. Como tratamento, foi gerada a coluna sinalizadora flag_sem_declaracao_financeira (booleana) para isolar esses 36,1% em análises de custo e evitar a subestimação da média de prejuízos.

In [ ]:
import pandas as pd
import numpy as np

def diagnostico_completude(df, colunas_vitais):
    total_linhas = len(df)
    # 1. Completude Sintática (Nulos Físicos)
    nulos_abs = df[colunas_vitais].isnull().sum()
    nulos_pct = (nulos_abs / total_linhas) * 100

    # 2. Completude Semântica ("Falsos Zeros" Financeiros)
    zerados_mas_com_status = df[
        (df['DM_total_danos_materiais'] == 0) &
        (df['PEPL_total_publico'] == 0)
    ]
    pct_falsos_zeros = (len(zerados_mas_com_status) / total_linhas) * 100

    # Adicionando a flag ao DataFrame
    df['flag_sem_declaracao_financeira'] = ((df['DM_total_danos_materiais'] == 0) &
                                            (df['PEPL_total_publico'] == 0))

    # Relatório Sintético
    df_completude = pd.DataFrame({
        'Nulos_Sintaticos_Abs': nulos_abs,
        'Nulos_Sintaticos_Pct (%)': nulos_pct.round(2)
    })

    print("=== DIAGNÓSTICO DE COMPLETUDE ===")
    print(df_completude)
    print(f"\n[Alerta Semântico] Registros com R$ 0,00 em Danos Materiais e Públicos: "
          f"{len(zerados_mas_com_status)} ocorrências ({pct_falsos_zeros:.2f}% do dataset).")
    return df, df_completude, zerados_mas_com_status

# Execução no recorte do Colab
colunas_vitais = [
    'Cod_IBGE_Mun',
    'Nome_Municipio',
    'Sigla_UF',
    'Data_Evento',
    'Status',
    'descricao_tipologia',
    'Protocolo_S2iD',
    'DM_total_danos_materiais',
    'PEPL_total_publico',
    'DH_MORTOS',
    'DH_DESALOJADOS',
    'DH_total_danos_humanos_diretos'
]
df_recorte, res_completude, df_falsos_zeros = diagnostico_completude(df_recorte, colunas_vitais)

##2. Diagnóstico de Qualidade: Consistência Interna

*O que foi avaliado:* A coerência matemática e lógica entre diferentes colunas do mesmo registro, validando o cumprimento das regras de negócio do S2iD.

🛠️ **Entendendo as Novas Colunas Criadas**

 Para garantir um dataset auditável e reproduzível sem excluir dados brutos, foram incorporadas **3 novas colunas** ao dataset. Abaixo explicamos a função de cada uma para orientar a equipe de pesquisa:

 1. 'DH\_total\_calculado' *(Tipo: Numérico)*

 O que é: O resultado exato da soma matemática realizada via código: \`DH\_MORTOS\` + \`DH\_DESALOJADOS\` + \`DH\_DESABRIGADOS\` + \`DH\_ENFERMOS\` + \`DH\_DESAPARECIDOS\`.

 **Por que foi criada:** Em 6,13% dos casos, o total oficial da base original continha erros de digitação/consolidação municipal. Esta coluna garante precisão matemática auditada.
 **Como usar na prática:** Utilize \`DH\_total\_calculado\` como a métrica principal para qualquer cálculo, estatística descritiva ou gráfico sobre o total de vítimas humanas.

 2. 'flag\_inconsistencia\_soma\_humana\' *(Tipo: Booleano - True/False)*

 **O que é:** Etiqueta que marca \`True\` exatamente nas **606 linhas (6,13%)** onde a conta original não fechou, e \`False\` onde a conta estava perfeita.

 **Por que foi criada:** Dá transparência ao erro sem deletar informações. Permite rastrear exatamente quais registros apresentavam desasincronia nos cadastros municipais.
 **Como usar na prática:** Caso a pesquisa exija um filtro ultra-rigoroso focado apenas em registros sem divergências cadastrais, basta filtrar a base por \`df[df['flag\_inconsistencia\_soma\_humana'] == False]\`.

 3. 'flag\_reconhecido\_sem\_impacto\' *(Tipo: Booleano - True/False)*

 **O que é:** Etiqueta que marca \`True\` nas **177 linhas (1,79%)** de desastres homologados pelo Governo Federal, mas que não possuem nenhum dano humano ou financeiro cadastrado no sistema.

 **Por que foi criada:** Separa a dimensão \*\*administrativa/legal\*\* (o reconhecimento da emergência) da dimensão \*\*estatística de impacto

 **Como usar na prática:** Ao calcular o prejuízo financeiro médio ou a média de afetados por evento reconhecido, recomenda-se filtrar esses 177 casos (\`flag\_reconhecido\_sem\_impacto == False\`) para evitar distorções nas médias.

In [ ]:
import pandas as pd
import numpy as np

def diagnostico_e_flags_consistencia(df):
    total_linhas = len(df)

    # 1. Calculando a soma real das subcategorias humanas
    # Correção (09/10): o total oficial do Atlas soma 6 categorias, incluindo os FERIDOS.
    # Sem eles apareciam 606 falsas inconsistências (ex.: Três Rios, 402 calculado x 432 oficial = 30 feridos).
    soma_humanos_calculada = (
        df['DH_MORTOS'].fillna(0) +
        df['DH_FERIDOS'].fillna(0) +
        df['DH_DESALOJADOS'].fillna(0) +
        df['DH_DESABRIGADOS'].fillna(0) +
        df['DH_ENFERMOS'].fillna(0) +
        df['DH_DESAPARECIDOS'].fillna(0)
    )

    # Criando a Coluna Derivada Auditada
    df['DH_total_calculado'] = soma_humanos_calculada

    # FLAG 1: Sinaliza se a soma não bate com o total oficial
    df['flag_inconsistencia_soma_humana'] = (soma_humanos_calculada != df['DH_total_danos_humanos_diretos'].fillna(0))

    # FLAG 2: Sinaliza se foi reconhecido mas não tem nenhum impacto
    sem_impacto_algum = (
        (df['DM_total_danos_materiais'] == 0) &
        (df['PEPL_total_publico'] == 0) &
        (df['DH_total_danos_humanos_diretos'] == 0)
    )
    df['flag_reconhecido_sem_impacto'] = (df['Status'] == 'Reconhecido') & sem_impacto_algum

    # Quantificação
    qtd_inc_soma = df['flag_inconsistencia_soma_humana'].sum()
    qtd_inc_status = df['flag_reconhecido_sem_impacto'].sum()

    relatorio = pd.DataFrame({
        'Regra_Consistencia': [
            'Soma de Danos Humanos Diretos == Total Oficial',
            'Desastre Reconhecido sem Impactos Declarados'
        ],
        'Inconsistencias_Detectadas': [qtd_inc_soma, qtd_inc_status],
        'Impacto_Percentual (%)': [
            round((qtd_inc_soma / total_linhas) * 100, 2),
            round((qtd_inc_status / total_linhas) * 100, 2)
        ]
    })

    print("=== DIAGNÓSTICO E AUDITORIA DE CONSISTÊNCIA INTERNA ===")
    print(relatorio.to_string(index=False))
    print("\n[Colunas Criadas no Dataset]:")
    print(" - 'DH_total_calculado': Soma auditada das subcategorias humanas.")
    print(f" - 'flag_inconsistencia_soma_humana': True para os {qtd_inc_soma} registros com divergência.")
    print(f" - 'flag_reconhecido_sem_impacto': True para os {qtd_inc_status} registros sem danos declarados.")
    return df, relatorio

# Execução no seu recorte
df_recorte, res_consistencia = diagnostico_e_flags_consistencia(df_recorte)


## 3\. Diagnóstico de Qualidade: Unicidade e Duplicidade

**🛠️ Decisão Metodológica e Tratamento**

**Baixa Incidência:** A duplicidade semântica afeta apenas 0,55% do dataset (54 registros em 9.890), confirmando a alta integridade cadastral da base.

**Tratamento Não Destrutivo:** Em vez de excluir diretamente esses registros, a coluna booleana `flag_duplicata_semantica` foi mantida como etiqueta. Para análises de somatória de prejuízos e contagem de eventos, a recomendação é consolidar essas duplicatas mantendo o registro de maior `Status` ou de maior valor de dano informado.




In [ ]:
def diagnostico_unicidade(df):
    total_linhas = len(df)

    # 1. Duplicidade por Chave Primária (Protocolo S2iD)
    # keep=False marca todas as duplicatas (incluindo a primeira ocorrência)
    df['flag_duplicata_protocolo'] = df.duplicated(subset=['Protocolo_S2iD'], keep=False)
    dup_protocolo = df['flag_duplicata_protocolo'].sum()

    # 2. Duplicidade Semântica (Mesmo Município + Mesma Data + Mesma Tipologia)
    chave_semantica = ['Cod_IBGE_Mun', 'Data_Evento', 'descricao_tipologia']
    df['flag_duplicata_semantica'] = df.duplicated(subset=chave_semantica, keep=False)
    dup_semanticos = df['flag_duplicata_semantica'].sum()

    relatorio_unicidade = pd.DataFrame({
        'Criterio_Unicidade': ['Chave Primária (Protocolo_S2iD)', 'Chave Semântica (IBGE + Data + Tipologia)'],
        'Registros_Duplicados': [dup_protocolo, dup_semanticos],
        'Percentual_Afetado (%)': [
            (dup_protocolo / total_linhas) * 100,
            (dup_semanticos / total_linhas) * 100
        ]
    })

    print("=== DIAGNÓSTICO DE UNICIDADE ===")
    print(relatorio_unicidade.to_string(index=False))
    return df, relatorio_unicidade

df_recorte, res_unicidade = diagnostico_unicidade(df_recorte)

##4.Diagnóstico de Qualidade: Acurácia, Validade e Outliers

**O que foi avaliado:** A plausibilidade numérica dos dados e a identificação de discrepâncias extremas (outliers severos) em prejuízos financeiros e danos humanos, utilizando o critério estatístico da amplitude interquartil severa (\\\\(Q3 + 3 \\times IQR\\\\)).

**Top 3 Maiores Eventos em Prejuízo Material Registrados**
 1. Maceió / AL (16/03/2021) - Movimento de Massa: **R$$ 7.651.464.416**(Crise de afundamento de solo urbanizado)

 2. São Leopoldo / RS (27/04/2024) - Chuvas Intensas: **R$4.823.240.240,88** (Enchentes históricas do Rio Grande do Sul).

 3. Pedro Leopoldo / MG (09/01/2022) - Chuvas Intensas: **R$ 1.184.366.614,71** (Inundações severas do início de 2022 em MG)

### Decisão Metodológica e Novas Colunas

Para viabilizar tanto a análise do custo total do país quanto o estudo de padrões típicos de eventos municipais recorrentes, foram incorporadas duas sinalizações booleanas:

1. **`flag_outlier_danos_materiais` (946 registros / 9,57%):**
   - Etiqueta `True` em desastres com prejuízos superiores a R\$ 5,31 milhões.
   - **Aplicação prática:** Permite calcular a **mediana** ou filtrar a base (`flag_outlier_danos_materiais == False`) para estimar o prejuízo médio típico sem a distorção dos mega-eventos de bilhões de reais.

2. **`flag_outlier_humanos` (1.456 registros / 14,72%):**
   - Etiqueta `True` em ocorrências com mais de 280 pessoas afetadas.
   - **Aplicação prática:** Permite separar eventos locais pequenos de desastres socioambientais de escala regional.

```

In [ ]:
import pandas as pd
import numpy as np

def diagnostico_e_flags_outliers(df):
    total_linhas = len(df)

    # 1. Análise de Outliers Severos em Danos Materiais (Regra do 3 * IQR)
    dm = df['DM_total_danos_materiais'].dropna()
    q1_dm = dm.quantile(0.25)
    q3_dm = dm.quantile(0.75)
    iqr_dm = q3_dm - q1_dm
    limite_sup_dm = q3_dm + (3.0 * iqr_dm)

    # Criando a FLAG para Danos Materiais Extremos
    df['flag_outlier_danos_materiais'] = df['DM_total_danos_materiais'] > limite_sup_dm
    qtd_outliers_dm = df['flag_outlier_danos_materiais'].sum()

    # 2. Análise de Outliers Severos em Danos Humanos Totais (Regra do 3 * IQR)
    dh = df['DH_total_calculado'].dropna()
    q1_dh = dh.quantile(0.25)
    q3_dh = dh.quantile(0.75)
    iqr_dh = q3_dh - q1_dh
    limite_sup_dh = q3_dh + (3.0 * iqr_dh)

    # Criando a FLAG para Afetados Humanos Extremos
    df['flag_outlier_humanos'] = df['DH_total_calculado'] > limite_sup_dh
    qtd_outliers_dh = df['flag_outlier_humanos'].sum()

    # Tabela Resumo
    relatorio_outliers = pd.DataFrame({
        'Métrica_Analisada': ['Danos Materiais (R$)', 'Total de Afetados Humanos (Pessoas)'],
        'Corte_Estatístico_Severo (3*IQR)': [f"R$ {limite_sup_dm:,.2f}", f"{limite_sup_dh:,.0f} pessoas"],
        'Qtd_Outliers_Detectados': [qtd_outliers_dm, qtd_outliers_dh],
        'Impacto_Percentual (%)': [
            round((qtd_outliers_dm / total_linhas) * 100, 2),
            round((qtd_outliers_dh / total_linhas) * 100, 2)
        ]
    })

    print("=== DIAGNÓSTICO DE ACURÁCIA E OUTLIERS ===")
    print(relatorio_outliers.to_string(index=False))

    print("\n[Top 3 Maiores Eventos em Prejuízo Material]:")
    top3_dm = df.sort_values(by='DM_total_danos_materiais', ascending=False)[
        ['Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'descricao_tipologia', 'DM_total_danos_materiais']
    ].head(3)
    print(top3_dm.to_string(index=False))

    return df, relatorio_outliers

# Execução no seu recorte
df_recorte, res_outliers = diagnostico_e_flags_outliers(df_recorte)


## 5. Diagnóstico de Qualidade: Atualidade e Cobertura Temporal

**O que foi avaliado:** A integridade da janela temporal da pesquisa (2021 a 2025), a ausência de erros na conversão de datas e a distribuição anual das ocorrências de desastres hidrológicos.

📅 Distribuição das Ocorrências por Ano de Registro

| Ano do Evento | Quantidade de Ocorrências | Proporção no Dataset (%) | Destaque Histórico do Período |
| :---: | :---: | :---: | :--- |
| **2021** | 1.334 | 13,49% | Início do recorte da pesquisa. |
| **2022** | 2.777 | 28,08% | Ano com maior volume de decretações hidrológicas no país. |
| **2023** | 2.361 | 23,87% | Manutenção de alto volume de eventos pluviométricos. |
| **2024** | 1.739 | 17,58% | Eventos de grande escala no Sul do país. |
| **2025** | 1.679 | 16,98% | Fechamento do período de amostragem. |
| **TOTAL** | **9.890** | **100,00%** | **Série temporal contínua e sem lacunas.** |

#🛠️ Decisão Metodológica

**Validação de Calendário:** A ausência de falhas na parsing de datas confirma que a ordenação cronológica e as análises sazonais (mensais e anuais) podem ser conduzidas sem perda de registros por erro de formatação.

In [ ]:
def diagnostico_atualidade(df, coluna_data):
    total_linhas = len(df)
    # Converte a coluna de data para datetime, forçando erros para NaN
    datas_dt = pd.to_datetime(df[coluna_data], format='%d/%m/%Y', errors='coerce')

    # 1. Datas Inválidas/Incompatíveis (que não puderam ser convertidas)
    datas_invalidas = datas_dt.isnull().sum()

    # 2. Datas Fora do Escopo (2021-2025)
    # Usamos o `dt.year` para extrair o ano e verificar se está fora do intervalo
    fora_do_escopo = ((datas_dt.dt.year < 2021) | (datas_dt.dt.year > 2025)).sum()

    # 3. Distribuição Anual de Ocorrências
    distribuicao_anual = datas_dt.dt.year.value_counts().sort_index().to_dict()

    relatorio = pd.DataFrame({
        'Métrica_Temporal': ['Datas Inválidas/Incompatíveis', 'Datas Fora do Escopo (2021-2025)'],
        'Quantidade': [datas_invalidas, fora_do_escopo],
        'Percentual (%)': [
            (datas_invalidas / total_linhas) * 100,
            (fora_do_escopo / total_linhas) * 100
        ]
    })

    print("=== DIAGNÓSTICO DE ATUALIDADE E COBERTURA TEMPORAL ===")
    print(relatorio.to_string(index=False))
    print(f"\nDistribuição de Ocorrências por Ano: {distribuicao_anual}")
    return relatorio

res_atualidade = diagnostico_atualidade(df_recorte, 'Data_Evento')

##6. Diagnóstico de Qualidade: Conformidade de Formato e Padronização
**O que foi avaliado:** Aderência estrita dos campos de identificação espacial e administrativa aos padrões oficiais (Expressões Regulares / Regex), verificando a estrutura do Código IBGE de 7 dígitos, siglas de UF e ausência de caracteres/espaços invisíveis.

In [ ]:
import re

def diagnostico_conformidade_formato(df):
    total_linhas = len(df)

    # 1. Código IBGE (Exatamente 7 dígitos numéricos)
    padrao_ibge = re.compile(r'^\d{7}$', re.IGNORECASE)
    # Primeiro, converte para string para lidar com possíveis nulos ou tipos mistos
    ibge_inconformes = ~df['Cod_IBGE_Mun'].astype(str).apply(lambda x: bool(padrao_ibge.match(x)))

    # 2. Espaços invisíveis em Municípios (nas pontas)
    nomes_originais = df['Nome_Municipio'].astype(str)
    espacos_extras = (nomes_originais != nomes_originais.str.strip())

    # 3. Sigla UF (2 letras maiúsculas)
    padrao_uf = re.compile(r'^[A-Z]{2}$', re.IGNORECASE)
    uf_inconformes = ~df['Sigla_UF'].astype(str).apply(lambda x: bool(padrao_uf.match(x)))

    relatorio_formato = pd.DataFrame({
        'Regra_Formato': [
            'Cod_IBGE_Mun (7 dígitos numéricos)',
            'Nome_Municipio (sem espaços extras nas pontas)',
            'Sigla_UF (2 letras maiúsculas)'
        ],
        'Inconformidades': [ibge_inconformes.sum(), espacos_extras.sum(), uf_inconformes.sum()],
        'Percentual_Afetado (%)': [
            (ibge_inconformes.sum() / total_linhas) * 100,
            (espacos_extras.sum() / total_linhas) * 100,
            (uf_inconformes.sum() / total_linhas) * 100
        ]
    })

    print("=== DIAGNÓSTICO DE CONFORMIDADE DE FORMATO (REGEX) ===")
    print(relatorio_formato.to_string(index=False))
    return relatorio_formato

res_formato = diagnostico_conformidade_formato(df_recorte)


# 📊 Tabela Consolidada de Diagnóstico de Qualidade de Dados (S2iD 2021–2025)

| Dimensão | Problema Diagnosticado | % Afetado (Qtd.) | Flag / Coluna Criada | Ação Recomendada |
| :--- | :--- | :---: | :--- | :--- |
| **1. Completude** | **Sintática:** Nulos em colunas vitais | **0,00%** *(0)* | — | Nenhuma (estrutura 100% íntegra). |
| **1. Completude** | **Semântica:** "Falsos Zeros" (R\$ 0,00 em danos) | **36,10%** *(3.567)* | `flag_sem_declaracao_financeira` | Preservar dados; isolar os 36,1% em cálculos de custo médio. |
| **2. Consistência** | **Aritmética:** Divergência na soma de humanos | **6,13%** *(606)* | `DH_total_calculado`<br />`flag_inconsistencia_soma_humana` | Adotar `DH_total_calculado` como métrica oficial auditada. |
| **2. Consistência** | **Regra:** Status "Reconhecido" sem impacto | **1,79%** *(177)* | `flag_reconhecido_sem_impacto` | Filtrar os 177 registros em estudos de valoração de danos. |
| **3. Unicidade** | **Chave Primária:** Duplicidade no protocolo | **0,00%** *(0)* | `flag_duplicata_protocolo` | Integridade do banco mantida. |
| **3. Unicidade** | **Semântica:** Mesma ocorrência cadastrada 2x | **0,55%** *(54)* | `flag_duplicata_semantica` | Manter etiqueta; consolidar pelo maior dano/status antes de somar. |
| **4. Acurácia** | **Outliers Financeiros:** Prejuízos &gt; R\$ 5,31 mi | **9,57%** *(946)* | `flag_outlier_danos_materiais` | Usar mediana ou filtrar a flag para estimar o custo típico. |
| **4. Acurácia** | **Outliers Humanos:** Vítimas &gt; 280 pessoas | **14,72%** *(1.456)* | `flag_outlier_humanos` | Separar eventos locais de catástrofes de escala regional. |
| **5. Atualidade** | **Série Temporal:** Datas inválidas/fora do escopo | **0,00%** *(0)* | — | Série contínua e válida em todo o período (2021–2025). |
| **6. Conformidade** | **Padronização:** Erros de Regex (IBGE/UF) | **0,00%** *(0)* | — | Chave geográfica 100% pronta para cruzamentos GIS/IBGE. |


## Exportação do Dataset Auditado

**Descrição:** Exportação final da base de dados de desastres hidrológicos (2021–2025) contendo os dados brutos sanitizados e todas as **8 colunas derivadas de auditoria e flags booleanas** criadas durante o diagnóstico metodológico:
- **Colunas Criadas:**

1. \`flag\_sem\_declaracao\_financeira\` (Completude)
2. \`DH\_total\_calculado\` (Consistência Aritmética)
3. \`flag\_inconsistencia\_soma\_humana\` (Consistência Aritmética)
4. \`flag\_reconhecido\_sem\_impacto\` (Consistência de Regra)
5. \`flag\_duplicata\_protocolo\` (Unicidade)
6. \`flag\_duplicata\_semantica\` (Unicidade)
7. \`flag\_outlier\_danos\_materiais\` (Acurácia)
8. \`flag\_outlier\_humanos\` (Acurácia)

**Parâmetros de Saída:** Formato CSV, separador \`;\`, codificação \`UTF-8-SIG\` e preservação total das 9.890 ocorrências.

In [ ]:
import pandas as pd
try:
    from google.colab import files
except ImportError:
    files = None

# 1. Definindo a lista exata de colunas que vão para o arquivo final
colunas_para_exportar = [
    # --- Colunas Vitais Base ---
    'Cod_IBGE_Mun',
    'Nome_Municipio',
    'Sigla_UF',
    'Data_Evento',
    'Status',
    'descricao_tipologia',
    'Protocolo_S2iD',
    'DM_total_danos_materiais',
    'PEPL_total_publico',

    # --- Novas Colunas Auditadas e Flags Criadas ---
    'DH_total_calculado',               # Substitui o total de danos humanos original (auditado e correto)
    'flag_sem_declaracao_financeira',    # Completude Semântica
    'flag_inconsistencia_soma_humana',   # Consistência Aritmética
    'flag_reconhecido_sem_impacto',      # Consistência de Regra
    'flag_duplicata_protocolo',         # Unicidade de Chave
    'flag_duplicata_semantica',         # Unicidade Semântica
    'flag_outlier_danos_materiais',     # Acurácia de Danos Materiais
    'flag_outlier_humanos'              # Acurácia de Afetados Humanos
]

# 2. Filtrando o DataFrame para conter SOMENTE as colunas selecionadas acima
df_exportar = df_recorte[colunas_para_exportar]

# 3. Exportando para CSV
nome_arquivo_csv = "S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv"

df_exportar.to_csv(
    nome_arquivo_csv,
    index=False,
    encoding='utf-8-sig',
    sep=';'
)

print(f"✅ Arquivo enxuto '{nome_arquivo_csv}' gerado com sucesso!")
print(f"📊 Total de linhas: {len(df_exportar)} | Total de colunas filtradas: {len(df_exportar.columns)}")

# 4. Download automático no navegador
if files: files.download(nome_arquivo_csv)

---

# 4. Cruzamento das bases do Observatório

**Notebook de origem:** `Notebook/Cruzamento_Bases_Observatorio.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

# Cruzamento das Bases — Observatório do Clima

**Squad Observatório do Clima – Grande Desafio DataConnect Solutions**

Este notebook faz o JOIN final das três bases já tratadas:

1. `sinisa_limpo.csv` — base do módulo Águas Pluviais do SINISA (5.570 municípios, tabela mestra).
2. `CAPAG-2025_Tratamento.csv` — capacidade de pagamento (nota CAPAG) por município.
3. `S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv` — histórico de desastres hidrológicos do Atlas (1 linha por desastre).

Saída: `dataset_mestre_observatorio.csv`, 1 linha por município, com os 5.570
municípios originais do SINISA preservados (left join).

## 0. Configuração do ambiente e upload dos arquivos

In [ ]:
import glob
import pandas as pd

pd.set_option("display.max_columns", 50)

try:
    from google.colab import files
    print("Ambiente Colab detectado. Selecione os 3 CSVs abaixo:")
    uploaded = files.upload()
except ImportError:
    print("Ambiente não é o Colab: assumindo que os arquivos já estão no diretório atual.")

def encontrar_arquivo(padrao):
    candidatos = glob.glob(padrao)
    if not candidatos:
        raise FileNotFoundError(f"Não encontrei nenhum arquivo correspondente a '{padrao}'.")
    if len(candidatos) > 1:
        print(f"Aviso: mais de um arquivo bate com '{padrao}': {candidatos}. Usando o primeiro.")
    return candidatos[0]

CAMINHO_SINISA = encontrar_arquivo("sinisa_limpo.csv")
CAMINHO_CAPAG = encontrar_arquivo("capag_nacional_tratada.csv")  # CAPAG set/2026 (Laura, 08/10)
CAMINHO_S2ID = encontrar_arquivo("*S2iD*.csv")

print("SINISA:", CAMINHO_SINISA)
print("CAPAG:", CAMINHO_CAPAG)
print("S2iD:", CAMINHO_S2ID)

## 1. Leitura blindada

Forçamos as colunas de código IBGE a `str` direto no `read_csv` — evita a
"maldição do decimal" (`3302502` virar `3302502.0`) e qualquer perda de zero
à esquerda.

Dois detalhes que os arquivos brutos exigem:
- Os três arquivos têm **BOM UTF-8** (`\ufeff`) no início — por isso usamos
  `encoding='utf-8-sig'`, senão o nome da primeira coluna de cada arquivo sai
  como `'\ufeffCod_IBGE_7'` em vez de `'Cod_IBGE_7'` e qualquer referência
  direta à coluna quebra silenciosamente.
- O S2iD usa `;` como separador, os outros dois usam `,`.

In [ ]:
df_sinisa = pd.read_csv(
    CAMINHO_SINISA,
    dtype={"Cod_IBGE_7": str},
    encoding="utf-8-sig",
)

df_capag = pd.read_csv(
    CAMINHO_CAPAG,
    dtype={"cod_ibge": str},
    encoding="utf-8-sig",
)

df_s2id = pd.read_csv(
    CAMINHO_S2ID,
    sep=";",
    dtype={"Cod_IBGE_Mun": str},
    encoding="utf-8-sig",
)

print("df_sinisa:", df_sinisa.shape, "| colunas:", list(df_sinisa.columns))
print("df_capag: ", df_capag.shape, "| colunas:", list(df_capag.columns))
print("df_s2id:  ", df_s2id.shape, "| colunas:", list(df_s2id.columns))

# Trava de versão: a CAPAG tem que ser a de setembro/2026
contagem_capag = df_capag["capag"].value_counts()
print("\nNotas CAPAG:", contagem_capag.to_dict())
assert contagem_capag.get("C", 0) == 2274 and contagem_capag.get("D", 0) == 20, "Esta CAPAG não é a de setembro/2026!"

### 1.1 Validações rápidas das chaves

Antes de cruzar, confirmamos que as colunas de código IBGE vieram como texto,
com 7 dígitos, sem duplicatas (nas bases que devem ser 1 linha por
município) e sem nulos.

In [ ]:
def checar_chave(df, coluna, nome_df, esperar_unico=True):
    print(f"--- {nome_df}.{coluna} ---")
    print(f"  dtype: {df[coluna].dtype}")
    print(f"  nulos: {df[coluna].isna().sum()}")
    tamanhos = df[coluna].dropna().str.len().value_counts()
    print(f"  tamanhos encontrados: {dict(tamanhos)}")
    duplicados = df[coluna].duplicated().sum()
    print(f"  duplicados: {duplicados}" + ("  <-- ESPERADO 0, VERIFIQUE!" if esperar_unico and duplicados else ""))
    print()

checar_chave(df_sinisa, "Cod_IBGE_7", "df_sinisa", esperar_unico=True)
checar_chave(df_capag, "cod_ibge", "df_capag", esperar_unico=True)
checar_chave(df_s2id, "Cod_IBGE_Mun", "df_s2id", esperar_unico=False)  # 1 linha por desastre, não por município

## 2. Tratamento pré-join

### 2.1 SINISA — flag `participou_sinisa`

`True` quando o município respondeu ao formulário (ou seja,
`mapeamento_area_risco` não é nulo), `False` quando não participou.

In [ ]:
df_sinisa["participou_sinisa"] = df_sinisa["mapeamento_area_risco"].notna()

print(df_sinisa["participou_sinisa"].value_counts())

### 2.2 S2iD — agregando de desastre (linha) para município (linha única)

A base traz uma linha por desastre; agregamos por `Cod_IBGE_Mun` antes do
join para não multiplicar linhas do SINISA no merge.

In [ ]:
df_s2id_agrupado = (
    df_s2id
    .groupby("Cod_IBGE_Mun")
    .agg(
        total_desastres=("Protocolo_S2iD", "count"),
        soma_danos_materiais=("DM_total_danos_materiais", "sum"),
        soma_prejuizo_publico=("PEPL_total_publico", "sum"),
        danos_humanos=("DH_total_calculado", "sum"),
    )
    .reset_index()
)

total_dh = int(df_s2id_agrupado["danos_humanos"].sum())
print("Municípios com pelo menos 1 desastre registrado:", df_s2id_agrupado.shape[0])
print("Total de danos humanos:", total_dh)
if total_dh != 3525510:
    print("AVISO: o esperado com os feridos é 3.525.510. Este CSV do Atlas ainda é a versão sem os feridos;"
          " a coluna danos_humanos fica subestimada até a Vitória subir o CSV corrigido.")
df_s2id_agrupado.head()

## 3. O grande cruzamento (left join)

`df_sinisa` é a tabela da esquerda — garante que os 5.570 municípios do
SINISA permanecem intactos no resultado final, com ou sem match nas outras
duas bases.

`validate="many_to_one"` em cada merge é uma checagem defensiva: se algum dia
a CAPAG ou o S2iD agrupado tiver uma chave duplicada, o merge quebra com erro
explícito em vez de silenciosamente multiplicar linhas do SINISA.

In [ ]:
linhas_antes = len(df_sinisa)

# --- SINISA + CAPAG (nota, critério C/D e as flags da decisão 1) ---
df_temp = df_sinisa.merge(
    df_capag[["cod_ibge", "capag", "criterio_capag_cd", "sem_nota_capag", "criterio_fiscal_priorizacao"]],
    left_on="Cod_IBGE_7",
    right_on="cod_ibge",
    how="left",
    validate="many_to_one",
)
print(f"Após merge com CAPAG: {len(df_temp)} linhas "
      f"({df_temp['cod_ibge'].notna().sum()} municípios encontraram match na CAPAG).")
assert len(df_temp) == linhas_antes, "O merge com a CAPAG alterou o número de linhas do SINISA!"

# --- (SINISA + CAPAG) + S2iD agrupado ---
df_final = df_temp.merge(
    df_s2id_agrupado,
    left_on="Cod_IBGE_7",
    right_on="Cod_IBGE_Mun",
    how="left",
    validate="many_to_one",
)
print(f"Após merge com S2iD agrupado: {len(df_final)} linhas "
      f"({df_final['Cod_IBGE_Mun'].notna().sum()} municípios com histórico de desastre).")
assert len(df_final) == linhas_antes, "O merge com o S2iD alterou o número de linhas do SINISA!"

### 3.1 Municípios sem linha na CAPAG

Registro de quem não encontrou par na CAPAG. Com a versão de abril eram 3: Brasília/DF (o Distrito Federal não é município e não tem CAPAG municipal), Fernando de Noronha/PE (distrito estadual) e Calçoene/AP (ausente no arquivo do Tesouro).

Atenção para a decisão 1 (filtro "quem não tem A ou B"): um filtro que só exclui A e B deixa passar quem não tem linha na CAPAG. A célula precisa decidir se "sem linha na CAPAG" conta como "sem nota".

In [ ]:
sem_capag = df_final[df_final["capag"].isna()][["Cod_IBGE_7", "Nome", "UF"]]
print(f"{len(sem_capag)} municípios sem linha na CAPAG:")
print(sem_capag.to_string(index=False))

## 4. Limpeza pós-join e exportação

- Remove `cod_ibge` e `Cod_IBGE_Mun` (chaves duplicadas do merge) — só
  `Cod_IBGE_7` permanece como chave primária.
- Município sem match no S2iD = 0 desastres, não nulo — preenchemos as 3
  colunas agregadas com `0`.

In [ ]:
df_final = df_final.drop(columns=["cod_ibge", "Cod_IBGE_Mun"])

colunas_s2id_agregadas = ["total_desastres", "soma_danos_materiais", "soma_prejuizo_publico", "danos_humanos"]
nulos_antes = df_final[colunas_s2id_agregadas].isna().sum()

df_final[colunas_s2id_agregadas] = df_final[colunas_s2id_agregadas].fillna(0)
df_final["total_desastres"] = df_final["total_desastres"].astype(int)

print("Nulos preenchidos com 0 em cada coluna agregada do S2iD:")
print(nulos_antes)

# Marcações do Atlas (decisão 2 usa tem_dano_atlas)
df_final["tem_registro_atlas"] = df_final["total_desastres"] > 0
df_final["tem_dano_atlas"] = (df_final["soma_danos_materiais"] + df_final["soma_prejuizo_publico"]) > 0
print("\nMunicípios com registro no Atlas:", int(df_final["tem_registro_atlas"].sum()))
print("Municípios com dano em R$ no Atlas:", int(df_final["tem_dano_atlas"].sum()))

In [ ]:
assert len(df_final) == 5570, f"Esperava 5.570 municípios, saíram {len(df_final)}."
assert df_final["Cod_IBGE_7"].duplicated().sum() == 0, "Chave primária Cod_IBGE_7 com duplicata!"
assert df_final[colunas_s2id_agregadas].isna().sum().sum() == 0, "Ainda há NaN nas colunas agregadas do S2iD."

print(f"Dataset final: {df_final.shape[0]} municípios, {df_final.shape[1]} colunas.")
print("Colunas:", list(df_final.columns))
df_final.head()

In [ ]:
CAMINHO_SAIDA = "dataset_mestre_observatorio.csv"
df_final.to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8-sig")

print(f"Exportado: {CAMINHO_SAIDA} ({len(df_final)} linhas, {df_final.shape[1]} colunas)")

try:
    from google.colab import files as colab_files
    colab_files.download(CAMINHO_SAIDA)
except ImportError:
    print("Fora do Colab: o arquivo foi salvo no diretório atual do notebook.")

---

# 5. Transformação e filtros — versão alternativa

**Notebook de origem:** `Transformacao_Filtros.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

**Transformação dos Dados e Análises Estatísticas**



IMPORTANTO AS BIBILHOTECAS

In [ ]:
import os
import pandas as pd
import numpy as np

# Leitura do Dataset Mestre (versão de 09/10: CAPAG set/2026 + colunas das decisões 1 e 2)
# 1º procura o arquivo local (pasta data/ do repositório ou upload no Colab);
# se não achar, lê direto do GitHub — assim roda no Colab sem precisar subir nada.
URL_GITHUB = 'https://raw.githubusercontent.com/Reginajose/Observatorio-Clima-Grande-desafio/main/data/dataset_mestre_observatorio.csv'
candidatos = ['data/dataset_mestre_observatorio.csv', '../data/dataset_mestre_observatorio.csv',
              'dataset_mestre_observatorio.csv', '/content/dataset_mestre_observatorio.csv']
caminho = next((c for c in candidatos if os.path.exists(c)), URL_GITHUB)

df_mestre = pd.read_csv(caminho, dtype={'Cod_IBGE_7': str}, encoding='utf-8-sig')

# Confere se é o mestre novo (09/10)
for col in ['criterio_fiscal_priorizacao', 'tem_dano_atlas', 'danos_humanos']:
    assert col in df_mestre.columns, f"Coluna {col} não encontrada: este é o dataset_mestre antigo. Baixe o novo do GitHub."

print(f"Dataset Mestre carregado: {df_mestre.shape[0]} municípios e {df_mestre.shape[1]} colunas.")

**1 - Aplicação dos Filtros do Funil**

In [ ]:


# 1. Base Inicial
f0 = df_mestre.copy()

# 2. Filtro 1: Participou do SINISA
f1 = f0[
    (f0['participou_sinisa'] == True)
    | (
        f0['participou_sinisa']
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(['TRUE', 'SIM', '1'])
    )
].copy()

# 3. Filtro 2: Domicílios em área de risco (> 0)
f1['qtd_domicilios_risco'] = pd.to_numeric(
    f1['qtd_domicilios_risco'], errors='coerce'
).fillna(0)
f1['pct_domicilios_risco'] = pd.to_numeric(
    f1['pct_domicilios_risco'], errors='coerce'
).fillna(0)
f2 = f1[f1['qtd_domicilios_risco'] > 0].copy()

# 4. Filtro 3: Fragilidade Fiscal / Sem Acesso a Crédito Bancário
# ATUALIZADO: Exclui apenas A, A+, B, B+ (Mantém C, D, n.d., n.e. e Nulos)
f2['capag_limpa'] = (
    f2['capag'].fillna('').astype(str).str.strip().str.upper()
)
f3 = f2[~f2['capag_limpa'].isin(['A', 'A+', 'B', 'B+'])].copy()

# 5. Filtro 4: Mapeamento de Risco Confirmado
f4 = f3[
    f3['mapeamento_area_risco']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(['SIM', 'S', '1', 'TRUE'])
].copy()

# 6. Filtro 5 — Decisão 2D (07/10): lista federal de Críticos + dano em R$ registrado no Atlas 2021-2025
f5 = f4[
    (f4['municipio_critico'].fillna('').astype(str).str.strip().str.upper() == 'SIM')
    & (f4['tem_dano_atlas'] == True)
].copy()

print('\n=== MÉTRICA DO FUNIL ATUALIZADA ===')
print(
    f'Total Inicial: {len(f0)} | SINISA: {len(f1)} | Com Risco: {len(f2)} |'
    f' CAPAG C/D/Sem Nota: {len(f3)} | Mapeados: {len(f4)} |'
    f' Elegíveis Finais (Crítico + dano no Atlas): {len(f5)}'
)

**Ordenação e Construção do Raking - TOP 20** -
(OPÇÃO 70% QTD / 30% PCT)

In [ ]:
f5['rank_qtd_pctile'] = f5['qtd_domicilios_risco'].rank(pct=True)
f5['rank_pct_pctile'] = f5['pct_domicilios_risco'].rank(pct=True)
f5['nota_70_30'] = (0.7 * f5['rank_qtd_pctile']) + (0.3 * f5['rank_pct_pctile'])

top_20 = f5.nlargest(20, 'nota_70_30').copy().reset_index(drop=True)
top_20['posicao'] = top_20.index + 1

In [ ]:
# Adicionando uma coluna de posição utilizando a variável correta (top_20).
top_20['posicao'] = top_20.index + 1

top_20.head(20)

DISTRIBUIÇÃO DA VERBA - OPÇÃO B (PROPORCIONAL TAMANHO DO MUNICÍPIO)

In [ ]:
VERBA_TOTAL = 150_000_000  # R$ 150 Milhões
PISO_MINIMO = 3_000_000  # R$ 3 Milhões (Piso)
TETO_MAXIMO = 15_000_000  # R$ 15 Milhões (Teto)

# Divisão proporcional às moradias com piso e teto (Decisão 4B):
# procura o valor por moradia (r) tal que a soma de min(max(r × moradias, piso), teto) dê exatamente R$ 150 mi.
# Assim todo município recebe o mesmo valor por moradia, exceto os que batem no piso ou no teto
# (e ninguém passa do teto nem fica abaixo do piso).
moradias = top_20['qtd_domicilios_risco']
lo, hi = 0.0, TETO_MAXIMO / moradias.min()
for _ in range(100):
  r = (lo + hi) / 2
  if (r * moradias).clip(PISO_MINIMO, TETO_MAXIMO).sum() < VERBA_TOTAL:
    lo = r
  else:
    hi = r
top_20['verba_ajustada'] = (r * moradias).clip(lower=PISO_MINIMO, upper=TETO_MAXIMO)

top_20['verba_alocada_m'] = (top_20['verba_ajustada'] / 1_000_000).round(2)

# Mapeamento do perfil fiscal
top_20['situacao_fiscal'] = (
    top_20['capag_limpa']
    .map({
        'C': 'Fragilidade Fiscal',
        'D': 'Alta Fragilidade Fiscal',
        'N.D.': 'Pendência Contábil (Sem Nota)',
        'N.E.': 'Não Elegível (Sem Nota)',
    })
    .fillna('Sem Nota/Pendente')
)

**Corte e cálculo "custo de não agir"**

In [ ]:
# Métrica de cobertura do orçamento
demais_elegiveis = f5[~f5['Cod_IBGE_7'].isin(top_20['Cod_IBGE_7'])].copy()

# Garantir conversão numérica das colunas do Atlas de Desastres
cols_atlas = ['total_desastres', 'soma_danos_materiais', 'danos_humanos']
for col in cols_atlas:
  top_20[col] = pd.to_numeric(top_20[col], errors='coerce').fillna(0)
  demais_elegiveis[col] = (
      pd.to_numeric(demais_elegiveis[col], errors='coerce').fillna(0)
  )

tabela_custo_nao_agir = pd.DataFrame([
    {
        'Grupo': '20 Selecionados (Top 20)',
        'Qtd Cidades': len(top_20),
        'Moradias em Risco (SINISA)': (
            f"{top_20['qtd_domicilios_risco'].sum():,.0f}".replace(',', '.')
        ),
        'Total de Desastres (2021-2025)': (
            f"{top_20['total_desastres'].sum():,.0f}".replace(',', '.')
        ),
        'Danos Materiais Totais (R$)': (
            f"R$ {top_20['soma_danos_materiais'].sum():,.2f}".replace(
                ',', 'X'
            )
            .replace('.', ',')
            .replace('X', '.')
        ),
        'Pessoas Afetadas (Danos Humanos)': (
            f"{top_20['danos_humanos'].sum():,.0f}".replace(',', '.')
        ),
    },
    {
        'Grupo': 'Demais Elegíveis (Fora do Top 20)',
        'Qtd Cidades': len(demais_elegiveis),
        'Moradias em Risco (SINISA)': (
            f"{demais_elegiveis['qtd_domicilios_risco'].sum():,.0f}".replace(
                ',', '.'
            )
        ),
        'Total de Desastres (2021-2025)': (
            f"{demais_elegiveis['total_desastres'].sum():,.0f}".replace(
                ',', '.'
            )
        ),
        'Danos Materiais Totais (R$)': (
            f"R$ {demais_elegiveis['soma_danos_materiais'].sum():,.2f}".replace(
                ',', 'X'
            )
            .replace('.', ',')
            .replace('X', '.')
        ),
        'Pessoas Afetadas (Danos Humanos)': (
            f"{demais_elegiveis['danos_humanos'].sum():,.0f}".replace(',', '.')
        ),
    },
])

print('=== TABELA: CUSTO DE NÃO AGIR ===')
display(tabela_custo_nao_agir)

Texte de Consistências

In [ ]:
moradias_atendidas = top_20['qtd_domicilios_risco'].sum()
moradias_totais_elegiveis = f5['qtd_domicilios_risco'].sum()
pct_cobertura = (moradias_atendidas / moradias_totais_elegiveis) * 100

print('\n=== TESTES DE CONSISTÊNCIA E VALIDAÇÃO DE REGRAS ===')

# Teste 1: Validação do Funil e Mapeamento
mapeamento_ok = (top_20['mapeamento_area_risco'] == 'Sim').sum()
print(
    f'✅ Teste 1 (Mapeamento de Risco): {mapeamento_ok} de 20 municípios no'
    ' Top 20 possuem Mapeamento de Risco confirmado.'
)

# Teste 2: Validação da Regra CAPAG (Exclusão de A, A+, B, B+)
capag_invalidas = top_20['capag_limpa'].isin(['A', 'A+', 'B', 'B+']).sum()
if capag_invalidas == 0:
  print(
      '✅ Teste 2 (Filtro Fiscal): 100% dos municípios no Top 20 possuem'
      ' restrição de acesso a crédito (CAPAG C, D ou Sem Nota).'
  )

# Teste 3: Cobertura de Moradias no Orçamento
print(
    f'✅ Teste 3 (Cobertura de Risco): O Top 20 protege'
    f' {moradias_atendidas:,.0f} moradias de um total de'
    f' {moradias_totais_elegiveis:,.0f} nos elegíveis ({pct_cobertura:.1f}% de'
    ' cobertura).'
)

# Teste 4: Fechamento Orçamentário (Verba = R$ 150 Mi)
verba_fechamento = top_20['verba_ajustada'].sum()
print(
    f'✅ Teste 4 (Fechamento Orçamentário): Soma total alocada no Top 20 = R$'
    f' {verba_fechamento/1e6:.2f} Mi (Meta: R$ 150,00 Mi).'
)

# Teste 5: Piso e teto respeitados (R$ 3 mi a R$ 15 mi por município)
fora_limite = (~top_20['verba_ajustada'].between(PISO_MINIMO - 1, TETO_MAXIMO + 1)).sum()
assert fora_limite == 0, f'{fora_limite} município(s) fora do piso/teto'
print(
    f'✅ Teste 5 (Piso e Teto): todos entre R$ 3 mi e R$ 15 mi; de R$'
    f" {(top_20['verba_ajustada'] / top_20['qtd_domicilios_risco']).min():.0f} a R$"
    f" {(top_20['verba_ajustada'] / top_20['qtd_domicilios_risco']).max():.0f} por moradia."
)

EXPORTAÇÃO DOS ARQUIVOS PARA O LOOKER STUDIO

In [ ]:
colunas_export = [
    'posicao',
    'Cod_IBGE_7',
    'Nome',
    'UF',
    'capag',
    'situacao_fiscal',
    'qtd_domicilios_risco',
    'verba_alocada_m',
    'total_desastres',
    'danos_humanos',
    'soma_danos_materiais',
]

tabela_export = top_20[colunas_export].copy()

# Exportação para CSV e Excel
csv_nome = 'Painel_v1_Top20_OpcaoB_ComSemNota.csv'
xlsx_nome = 'Painel_v1_Top20_OpcaoB_ComSemNota.xlsx'

tabela_export.to_csv(csv_nome, index=False, encoding='utf-8-sig')
tabela_export.to_excel(xlsx_nome, index=False)

print(
    f"\n🚀 Sucesso! Arquivos '{csv_nome}' e '{xlsx_nome}' prontos para o Looker"
    ' Studio.'
)

# Baixa os arquivos quando roda no Colab (fora dele, ficam salvos na pasta do notebook)
try:
  from google.colab import files
  files.download(csv_nome)
  files.download(xlsx_nome)
except ImportError:
  pass

---

# 6. Transformação e filtros — versão principal

**Notebook de origem:** `Notebook/Transformacao-Filtros.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

**Transformação dos Dados e Análises Estatísticas**



In [ ]:
import pandas as pd
import numpy as np

# Criando sub dataset´s para não estragar os dados que ja foram tratados e cruzados.

df_funil = pd.read_csv('/content/dataset_mestre_observatorio.csv').copy()

#Filtro 1
df_f1 = df_funil [df_funil[ 'participou_sinisa'] == True].copy ()

#Filtro 2
df_f2 = df_f1[df_f1['qtd_domicilios_risco'] > 0]. copy ()

#Filtro 3
df_f3  = df_f2[df_f2['capag'].isin(['C', 'D'])].copy ()

#Filtro 4
df_f4 = df_f3[df_f3['mapeamento_area_risco'].notna()].copy()

print(f"Total Inicial: {len(df_funil)}")
print(f"Apos filtro SINISA: {len(df_f1)}")
print(f"Com Domicílios em Risco: {len(df_f2)}")
print(f"Com CAPAG C/D: {len(df_f3)}")
print(f"Com Risco Mapeado (Universo ELegível): {len(df_f4)}")

**Ordenação e Construção do Raking - TOP 20**

In [ ]:
# Ordenando pelo príncipal de valor de impacto
# Aplicando Critérios de desempate

df_ranking = df_f4.sort_values(
    by=['qtd_domicilios_risco','total_desastres', 'soma_afetados'],
    ascending=[False, False, False]
).reset_index(drop=True)

df_ranking.head(20)

In [ ]:
# Adicionando uma coluna de posição.
df_ranking['posicao'] = df_ranking.index + 1

df_ranking.head(20)

In [ ]:
# Seleção inicial dos 20 primeiros municipois a receberem o finaciamento do Carlos.
top_20 = df_ranking.head(20).copy()



Verificar a posibilidade da distribuição ser realizada em função do tamanho do município - Amostra Polulacional.

Corte e cálculo "custo de não agir"

In [ ]:
# Metrica do custo da decisão
moradias_atendidas = top_20[qtd_domicilios_risco].sum()
moradias_elegiveis_totais = df_ranking['qtd_domicilios_risco'].sun()
moradias_fora = moradias_elegiveis_totais - moradias_atendidas


print(f"/nMoradias em risco atendidas pelos 150M: {moradias_atendidas}")
print(f"/nMoradias em risco não atendidas: {moradias_fora}

---

# 7. Ranking, funil e robustez

**Notebook de origem:** `Notebook/Analise_de_Raking_Funil_Robustez.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
# 1. Carregamento da base de dados
# Corrigido o separador de ';' para ',' pois a base utiliza vírgulas.
caminho_arquivo = '/content/drive/MyDrive/dataset_mestre_observatorio.csv'

print("Lendo o arquivo original...")
df_bruto = pd.read_csv(caminho_arquivo, sep=',', encoding='utf-8')

print(f"Base carregada com sucesso! O arquivo inteiro tem {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

### 🔍 Diagnóstico das Variáveis de Elegibilidade

Antes de aplicarmos os filtros finais, é importante inspecionar a qualidade, a presença de dados nulos e a formatação das principais colunas no dataset mestre. Isso garante que nenhum dado relevante seja excluído acidentalmente devido a diferenças de digitação ou representação de valores lógicos.

In [ ]:
# 1. Quantas linhas sua tabela tem antes dos filtros?
print("Total de municípios na tabela mestre:", len(df))

# 2. Como estão gravadas as colunas de filtro?
print("\nValores em 'participou_sinisa':", df['participou_sinisa'].unique())
print("Valores em 'mapeamento_area_risco':", df['mapeamento_area_risco'].unique())
print("Valores em 'capag':", df['capag'].unique())

# Diagnóstico extra: contagem de valores nulos ou filtros muito restritivos
print("\nMunicípios com participou_sinisa == 'Sim':", len(df[df['participou_sinisa'] == 'Sim']))
print("Municípios com mapeamento_area_risco == 'Sim':", len(df[df['mapeamento_area_risco'] == 'Sim']))
print("Municípios com qtd_domicilios_risco > 0:", len(df[df['qtd_domicilios_risco'] > 0]))

### 📊 Pipeline de Elegibilidade e Teste de Robustez dos Cenários

Este bloco de código consolida duas etapas fundamentais do nosso estudo:

1. **Tratamento e Aplicação de Filtros (Elegibilidade):** O algoritmo realiza a limpeza e padronização das bases, tratando valores nulos e convertendo colunas de texto e booleanos para aplicar de forma rigorosa os quatro critérios de elegibilidade (participação no **SINISA**, existência de **Mapeamento de Risco**, exclusão de **CAPAG** 'A' e 'B', e presença de **Domicílios em Risco**). Ao final, o diagnóstico exibe quantos municípios restaram em cada etapa até chegar aos **641 elegíveis**.
2. **Simulação de Pesos (Análise de Sensibilidade):** Com as cidades elegíveis definidas, o código simula de forma automatizada quatro cenários ponderando a escala absoluta (quantidade de domicílios em risco) e a vulnerabilidade local (percentual de domicílios em risco). A tabela de robustez gerada calcula o trade-off de moradias atendidas no Top 20 e o nível de sobreposição de municípios em relação ao Cenário Base.

In [ ]:
import pandas as pd
import numpy as np

# 1. Criar uma cópia de trabalho
df_test = df.copy()

# 2. Filtro 1: SINISA (Reconhece o valor lógico True)
filtro_sinisa = (df_test['participou_sinisa'] == True) | (df_test['participou_sinisa'].astype(str).str.strip().str.upper().isin(['TRUE', 'SIM', '1']))

# 3. Filtro 2: Mapeamento de Risco (Reconhece 'Sim')
filtro_mapeamento = df_test['mapeamento_area_risco'].fillna('').astype(str).str.strip().str.upper().isin(['SIM', 'S', '1', 'TRUE'])

# 4. Filtro 3: CAPAG (Exclui notas A, A+, B e B+ conforme Decisão 1B)
capag_limpo = df_test['capag'].fillna('').astype(str).str.strip().str.upper()
filtro_capag = ~capag_limpo.isin(['A', 'A+', 'B', 'B+'])

# 5. Filtro 4: Moradias em Risco (> 0)
df_test['qtd_domicilios_risco'] = pd.to_numeric(df_test['qtd_domicilios_risco'], errors='coerce').fillna(0)
df_test['pct_domicilios_risco'] = pd.to_numeric(df_test['pct_domicilios_risco'], errors='coerce').fillna(0)
filtro_risco = df_test['qtd_domicilios_risco'] > 0

# 6. Filtrar a lista final de elegíveis
df_elegiveis = df_test[filtro_sinisa & filtro_mapeamento & filtro_capag & filtro_risco].copy()

print("=== DIAGNÓSTICO DOS FILTROS ===")
print(f"Total na tabela mestre: {len(df_test)}")
print(f"1. Participou do SINISA (True): {len(df_test[filtro_sinisa])}")
print(f"2. Mapeamento de Risco (Sim): {len(df_test[filtro_mapeamento])}")
print(f"3. CAPAG Elegível (sem A, A+, B, B+): {len(df_test[filtro_capag])}")
print(f"4. Domicílios em Risco (>0): {len(df_test[filtro_risco])}")
print(f"--> ELEGÍVEIS FINAIS: {len(df_elegiveis)}")

# 7. Rodar o Teste de Robustez (Decisão 3)
if len(df_elegiveis) > 0:
    # Transforma em percentil (régua de 0.0 a 1.0)
    df_elegiveis['rank_qtd_pctile'] = df_elegiveis['qtd_domicilios_risco'].rank(pct=True)
    df_elegiveis['rank_pct_pctile'] = df_elegiveis['pct_domicilios_risco'].rank(pct=True)

    cenarios = [
        (1.0, 0.0, "100% Qtd / 0% Pct (Cenário Base)"),
        (0.7, 0.3, "70% Qtd / 30% Pct"),
        (0.5, 0.5, "50% Qtd / 50% Pct"),
        (0.0, 1.0, "0% Qtd / 100% Pct")
    ]

    top20_base_ids = set(df_elegiveis.nlargest(20, 'rank_qtd_pctile')['Cod_IBGE_7'])

    resultados = []
    for peso_qtd, peso_pct, nome_cenario in cenarios:
        df_elegiveis['nota_final'] = (peso_qtd * df_elegiveis['rank_qtd_pctile']) + (peso_pct * df_elegiveis['rank_pct_pctile'])
        top20_cenario = df_elegiveis.nlargest(20, 'nota_final')
        top20_cenario_ids = set(top20_cenario['Cod_IBGE_7'])

        permanecem = len(top20_cenario_ids.intersection(top20_base_ids))
        total_moradias = top20_cenario['qtd_domicilios_risco'].sum()

        resultados.append({
            'Cenário de Pesos': nome_cenario,
            'Permanecem do Top 20 Base': f"{permanecem} de 20",
            'Moradias Cobertas no Top 20': f"{total_moradias:,.0f}"
        })

    df_robustez = pd.DataFrame(resultados)
    print("\n=== TABELA DE ROBUSTEZ DA LISTA SELECIONADA ===")
    print(df_robustez.to_string(index=False))

### ⚙️ Detalhamento do Pipeline de Elegibilidade e Robustez

Este bloco de código executa as etapas lógicas cruciais para a consolidação das análises. Abaixo está o detalhamento de como cada parte do script opera:

---

#### 1. Criação do DataFrame de Trabalho
* O código cria uma cópia isolada (`df_test`) do DataFrame principal. Isso evita qualquer efeito colateral indesejado nas variáveis globais do seu ambiente.

#### 2. Normalização e Limpeza dos Filtros (Etapas 2 a 5)
* **SINISA (`filtro_sinisa`):** Garante a inclusão dos municípios participantes convertendo valores lógicos e strings de texto (`'Sim'`, `'True'`, `'1'`) de forma uniforme para maiúsculas e sem espaços adicionais.
* **Mapeamento de Risco (`filtro_mapeamento`):** Identifica e filtra municípios com planos e diagnósticos ativos, preenchendo eventuais nulos com vazios para evitar erros de comparação lógica.
* **CAPAG (`filtro_capag`):** Exclui de forma robusta e dinâmica as classificações de crédito mais favoráveis (`A`, `A+`, `B` e `B+`), aplicando o operador de negação (`~`).
* **Domicílios em Risco (`filtro_risco`):** Força a conversão das métricas absolutas e proporcionais para o tipo numérico, tratando dados inválidos como zero e garantindo que apenas cidades com vulnerabilidade real mapeada sejam mantidas.

#### 3. Diagnóstico Iterativo dos Filtros (Etapa 6)
* Imprime na tela um balanço passo a passo do funil de elegibilidade, permitindo auditar o impacto individual de cada um dos critérios definidos.

#### 4. Simulação de Sensibilidade por Percentil (Etapa 7)
* **Régua de Percentis:** Transforma as colunas brutas em percentis (`rank(pct=True)`), colocando os diferentes eixos (absoluto vs. proporcional) na mesma escala de comparação (de 0.0 a 1.0).
* **Notas Ponderadas:** Calcula e simula as notas finais dos municípios nos quatro cenários de peso definidos.
* **Tabela de Robustez:** Apresenta a contagem de moradias cobertas no Top 20 de cada cenário e compara o grau de permanência das cidades de alta prioridade frente ao modelo de referência absoluto.

In [ ]:
import pandas as pd
import numpy as np

# 1. Criar uma cópia de trabalho
df_test = df.copy()

# 2. Filtro 1: SINISA (Reconhece o valor lógico True)
filtro_sinisa = (df_test['participou_sinisa'] == True) | (df_test['participou_sinisa'].astype(str).str.strip().str.upper().isin(['TRUE', 'SIM', '1']))

# 3. Filtro 2: Mapeamento de Risco (Reconhece 'Sim')
filtro_mapeamento = df_test['mapeamento_area_risco'].fillna('').astype(str).str.strip().str.upper().isin(['SIM', 'S', '1', 'TRUE'])

# 4. Filtro 3: CAPAG (Exclui notas A, A+, B e B+ conforme Decisão 1B)
capag_limpo = df_test['capag'].fillna('').astype(str).str.strip().str.upper()
filtro_capag = ~capag_limpo.isin(['A', 'A+', 'B', 'B+'])

# 5. Filtro 4: Moradias em Risco (> 0)
df_test['qtd_domicilios_risco'] = pd.to_numeric(df_test['qtd_domicilios_risco'], errors='coerce').fillna(0)
df_test['pct_domicilios_risco'] = pd.to_numeric(df_test['pct_domicilios_risco'], errors='coerce').fillna(0)
filtro_risco = df_test['qtd_domicilios_risco'] > 0

# 6. Filtrar a lista final de elegíveis
df_elegiveis = df_test[filtro_sinisa & filtro_mapeamento & filtro_capag & filtro_risco].copy()

print("=== DIAGNÓSTICO DOS FILTROS ===")
print(f"Total na tabela mestre: {len(df_test)}")
print(f"1. Participou do SINISA (True): {len(df_test[filtro_sinisa])}")
print(f"2. Mapeamento de Risco (Sim): {len(df_test[filtro_mapeamento])}")
print(f"3. CAPAG Elegível (sem A, A+, B, B+): {len(df_test[filtro_capag])}")
print(f"4. Domicílios em Risco (>0): {len(df_test[filtro_risco])}")
print(f"--> ELEGÍVEIS FINAIS: {len(df_elegiveis)}")

# 7. Rodar o Teste de Robustez (Decisão 3)
if len(df_elegiveis) > 0:
    # Transforma em percentil (régua de 0.0 a 1.0)
    df_elegiveis['rank_qtd_pctile'] = df_elegiveis['qtd_domicilios_risco'].rank(pct=True)
    df_elegiveis['rank_pct_pctile'] = df_elegiveis['pct_domicilios_risco'].rank(pct=True)

    cenarios = [
        (1.0, 0.0, "100% Qtd / 0% Pct (Cenário Base)"),
        (0.7, 0.3, "70% Qtd / 30% Pct"),
        (0.5, 0.5, "50% Qtd / 50% Pct"),
        (0.0, 1.0, "0% Qtd / 100% Pct")
    ]

    top20_base_ids = set(df_elegiveis.nlargest(20, 'rank_qtd_pctile')['Cod_IBGE_7'])

    resultados = []
    for peso_qtd, peso_pct, nome_cenario in cenarios:
        df_elegiveis['nota_final'] = (peso_qtd * df_elegiveis['rank_qtd_pctile']) + (peso_pct * df_elegiveis['rank_pct_pctile'])
        top20_cenario = df_elegiveis.nlargest(20, 'nota_final')
        top20_cenario_ids = set(top20_cenario['Cod_IBGE_7'])

        permanecem = len(top20_cenario_ids.intersection(top20_base_ids))
        total_moradias = top20_cenario['qtd_domicilios_risco'].sum()

        resultados.append({
            'Cenário de Pesos': nome_cenario,
            'Permanecem do Top 20 Base': f"{permanecem} de 20",
            'Moradias Cobertas no Top 20': f"{total_moradias:,.0f}"
        })

    df_robustez = pd.DataFrame(resultados)
    print("\n=== TABELA DE ROBUSTEZ DA LISTA SELECIONADA ===")
    print(df_robustez.to_string(index=False))

In [ ]:
import pandas as pd

# 1. Garantir que as colunas numéricas do Atlas sejam números
cols_atlas = ['total_desastres', 'soma_danos_materiais', 'soma_afetados']
for col in cols_atlas:
    df_elegiveis[col] = pd.to_numeric(df_elegiveis[col], errors='coerce').fillna(0)

# 2. Separar os 20 Campeões (Cenário 70% Qtd / 30% Pct) dos demais elegíveis
df_elegiveis['nota_final'] = (0.7 * df_elegiveis['rank_qtd_pctile']) + (0.3 * df_elegiveis['rank_pct_pctile'])
top20 = df_elegiveis.nlargest(20, 'nota_final')
demais_elegiveis = df_elegiveis[~df_elegiveis['Cod_IBGE_7'].isin(top20['Cod_IBGE_7'])]

# 3. Montar o Resumo do "Custo de Não Agir"
tabela_custo_nao_agir = pd.DataFrame([
    {
        'Grupo': '20 Selecionados (Top 20)',
        'Qtd Cidades': len(top20),
        'Moradias em Risco (SINISA)': f"{top20['qtd_domicilios_risco'].sum():,.0f}",
        'Total de Desastres (2021-2025)': f"{top20['total_desastres'].sum():,.0f}",
        'Danos Materiais Totais (R$)': f"R$ {top20['soma_danos_materials'].sum() if 'soma_danos_materials' in top20.columns else top20['soma_danos_materiais'].sum():,.2f}",
        'Pessoas Afetadas (Danos Humanos)': f"{top20['soma_afetados'].sum():,.0f}"
    },
    {
        'Grupo': 'Demais Elegíveis (Fora do Top 20)',
        'Qtd Cidades': len(demais_elegiveis),
        'Moradias em Risco (SINISA)': f"{demais_elegiveis['qtd_domicilios_risco'].sum():,.0f}",
        'Total de Desastres (2021-2025)': f"{demais_elegiveis['total_desastres'].sum():,.0f}",
        'Danos Materiais Totais (R$)': f"R$ {demais_elegiveis['soma_danos_materials'].sum() if 'soma_danos_materials' in demais_elegiveis.columns else demais_elegiveis['soma_danos_materiais'].sum():,.2f}",
        'Pessoas Afetadas (Danos Humanos)': f"{demais_elegiveis['soma_afetados'].sum():,.0f}"
    }
])

print("=== TABELA: CUSTO DE NÃO AGIR ===")
print(tabela_custo_nao_agir.to_string(index=False))

### 💾 Exportação e Consolidação Multicritério do Universo de Elegíveis

Este bloco de código é responsável por estruturar o banco de dados final com o histórico completo de posições de todos os **641 municípios elegíveis** sob diferentes visões metodológicas. Veja o detalhamento do processo:

---

#### 1. Cálculo de Notas Simultâneas (Etapas 1 e 2)
* O script recalcula as notas individuais dos 4 cenários de pesos simulados (Base 100/0, Recomendado 70/30, Equilibrado 50/50 e Proporcional 0/100) utilizando as colunas normalizadas por percentil.

#### 2. Geração de Rankings Dinâmicos (Etapa 3)
* Utilizando a função `.rank(ascending=False, method='min')`, o script atribui a posição exata de cada município (do 1º ao 641º lugar) para cada um dos quatro modelos de pesos. Isso permite visualizar como um município sobe ou desce no ranking à medida que o foco político migra da escala absoluta de moradias para a escala de vulnerabilidade proporcional.

#### 3. Sinalização de Alta Prioridade (Etapa 4)
* Cria uma coluna booleana (`is_top20_recomendado`) que marca com `True` apenas as 20 primeiras cidades do cenário recomendado (70/30), servindo como filtro rápido para análises e integrações em ferramentas de mapas ou relatórios.

#### 4. Geração dos Arquivos de Saída (Etapas 5 e 6)
* O banco é ordenado de forma lógica a partir do ranking recomendado (70/30).
* Exporta o conjunto completo de dados e rankings para os formatos **CSV** (otimizado para sistemas de Geoprocessamento e bancos de dados) e **Excel** (formatado para leitura direta por gestores públicos), engatando em seguida o download automático no seu navegador.

In [ ]:
import pandas as pd
from google.colab import files

# 1. Garantir o cálculo da nota combinada (70/30) e selecionar os Top 20
df_elegiveis['nota_final'] = (0.7 * df_elegiveis['rank_qtd_pctile']) + (0.3 * df_elegiveis['rank_pct_pctile'])
top20 = df_elegiveis.nlargest(20, 'nota_final').copy()

# 2. Selecionar as colunas principais para relatório e geoprocessamento
colunas_desejadas = [
    'Cod_IBGE_7', 'Nome', 'UF',
    'qtd_domicilios_risco', 'pct_domicilios_risco',
    'nota_final', 'capag',
    'total_desastres', 'soma_danos_materiais', 'soma_afetados'
]

# Mantém apenas as colunas que efetivamente existem no seu DataFrame
colunas_presentes = [c for c in colunas_desejadas if c in top20.columns]
top20_export = top20[colunas_presentes].copy()

# Arredondar a nota final para 4 casas decimais
if 'nota_final' in top20_export.columns:
    top20_export['nota_final'] = top20_export['nota_final'].round(4)

# 3. Salvar os arquivos
top20_export.to_csv('top20_municipios_prioritarios.csv', index=False, encoding='utf-8-sig')
top20_export.to_excel('top20_municipios_prioritarios.xlsx', index=False)

print("=== LISTA DOS 20 MUNICÍPIOS CAMPEÕES ===")
if set(['Nome', 'UF', 'qtd_domicilios_risco']).issubset(top20_export.columns):
    print(top20_export[['Cod_IBGE_7', 'Nome', 'UF', 'qtd_domicilios_risco', 'nota_final']].to_string(index=False))

# 4. Disparar o download automático no Colab
files.download('top20_municipios_prioritarios.csv')
files.download('top20_municipios_prioritarios.xlsx')

### 🏁 Conclusão do Estudo de Priorização

Este caderno consolidou o desenvolvimento de uma metodologia transparente, auditável e balanceada para priorização de investimentos e apoio técnico municipal no âmbito do Observatório.

---

#### 📈 Síntese dos Principais Marcos Analíticos

1. **Filtros de Elegibilidade:** Reduziu o universo inicial de **5.570 municípios** para **641 habilitados** usando quatro regras institucionais concomitantes (Participação ativa no SINISA, mapeamento de risco local, exclusão de CAPAGs de alto crédito e presença real de moradias sob perigo climático).
2. **Teste de Sensibilidade:** Demonstrou numericamente o impacto de calibrar o foco estratégico. O cenário recomendado (**70% Quantidade / 30% Percentual de Risco**) provou ser robusto ao equilibrar a escala absoluta das grandes metrópoles com a vulnerabilidade relativa dos pequenos municípios, garantindo a proteção direta de **598.353 moradias** no Top 20.
3. **Custo de Não Agir:** Confirmou a eficiência fiscal do modelo de priorização, demonstrando que o Top 20 sugerido absorve preventivamente uma parcela significativa do passivo de desastres e perdas financeiras históricas.
4. **Arquivos Gerados:** Toda a inteligência geográfica e as posições de ranking foram consolidadas nos arquivos locais `top20_municipios_prioritarios` e `elegiveis_completo_cenarios` para alimentar diretamente painéis BI e mapas no GIS.

In [ ]:
import pandas as pd
from google.colab import files

# 1. Garantir os percentis das variáveis
df_elegiveis['rank_qtd_pctile'] = df_elegiveis['qtd_domicilios_risco'].rank(pct=True)
df_elegiveis['rank_pct_pctile'] = df_elegiveis['pct_domicilios_risco'].rank(pct=True)

# 2. Calcular as notas dos 4 cenários
df_elegiveis['nota_100_0'] = (1.0 * df_elegiveis['rank_qtd_pctile']) + (0.0 * df_elegiveis['rank_pct_pctile'])
df_elegiveis['nota_70_30'] = (0.7 * df_elegiveis['rank_qtd_pctile']) + (0.3 * df_elegiveis['rank_pct_pctile'])
df_elegiveis['nota_50_50'] = (0.5 * df_elegiveis['rank_qtd_pctile']) + (0.5 * df_elegiveis['rank_pct_pctile'])
df_elegiveis['nota_0_100'] = (0.0 * df_elegiveis['rank_qtd_pctile']) + (1.0 * df_elegiveis['rank_pct_pctile'])

# 3. Criar a Posição no Ranking (1º, 2º, 3º...) para cada cenário
df_elegiveis['posicao_100_0'] = df_elegiveis['nota_100_0'].rank(ascending=False, method='min').astype(int)
df_elegiveis['posicao_70_30'] = df_elegiveis['nota_70_30'].rank(ascending=False, method='min').astype(int)
df_elegiveis['posicao_50_50'] = df_elegiveis['nota_50_50'].rank(ascending=False, method='min').astype(int)
df_elegiveis['posicao_0_100'] = df_elegiveis['nota_0_100'].rank(ascending=False, method='min').astype(int)

# 4. Criar um marcador simples se o município é Top 20 no cenário recomendado (70/30)
df_elegiveis['is_top20_recomendado'] = df_elegiveis['posicao_70_30'] <= 20

# 5. Ordenar pelo ranking recomendado (70/30)
df_completo_export = df_elegiveis.sort_values(by='posicao_70_30').copy()

# 6. Salvar e realizar o download automático dos 641 elegíveis
df_completo_export.to_csv('elegiveis_completo_cenarios.csv', index=False, encoding='utf-8-sig')
df_completo_export.to_excel('elegiveis_completo_cenarios.xlsx', index=False)

print(f"Sucesso! Exportada a base completa com {len(df_completo_export)} municípios elegíveis e todos os cenários.")

files.download('elegiveis_completo_cenarios.csv')
files.download('elegiveis_completo_cenarios.xlsx')

---

# 8. Painel final e exportações

**Notebook de origem:** `Notebook/Painel_Final.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

# Painel final — lista de prioridade e valores de referência

**Squad Observatório do Clima – Grande Desafio DataConnect Solutions**

Este notebook gera **todos os números do painel e da apresentação** a partir do `dataset_mestre_observatorio.csv`, aplicando as decisões da célula (reunião de 19/10/2026):

| Decisão | Regra |
|---|---|
| Funil | SINISA respondido → moradias em risco > 0 → sem CAPAG A/B → risco mapeado → crítico e com dano no Atlas (318 elegíveis) |
| Lista oficial (A) | Os 20 com mais moradias em risco (desempate: desastres, depois pessoas afetadas) |
| Alternativa estudada (C) | Nota = 70% percentil do nº de moradias em risco + 30% percentil da parte da cidade em risco |
| Valores | Referência: mesmo R$ por moradia, piso R$ 3 mi, teto R$ 15 mi, total R$ 150 mi |
| Danos | Danos materiais + prejuízo econômico público (Atlas) |

**Saídas** (pasta `painel/`): os 3 CSVs que o Looker Studio usa (`painel_top20.csv`, `painel_funil.csv`, `painel_dentro_fora.csv`) e as tabelas de apoio (suplentes: os 298 demais elegíveis na ordem; alternativa C; trocas A × C; ranking dos 318).

## 0. Leitura

In [ ]:
import os
import numpy as np
import pandas as pd

URL_MESTRE = 'https://raw.githubusercontent.com/Reginajose/Observatorio-Clima-Grande-desafio/main/data/dataset_mestre_observatorio.csv'
URL_COORD = 'https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv'  # lat/long por código IBGE

# usa o arquivo local se o notebook estiver rodando dentro do repositório; senão baixa do GitHub
caminho = next((c for c in ['data/dataset_mestre_observatorio.csv', '../data/dataset_mestre_observatorio.csv'] if os.path.exists(c)), URL_MESTRE)
d = pd.read_csv(caminho, encoding='utf-8-sig')
for col in ['criterio_fiscal_priorizacao', 'tem_dano_atlas', 'participou_sinisa']:
    d[col] = d[col].astype(str).str.strip().str.upper().eq('TRUE')
d['danos_rs'] = d['soma_danos_materiais'] + d['soma_prejuizo_publico']   # danos = DM + PEPL
print(caminho, d.shape)

## 1. Funil: de 5.570 a 318 elegíveis

In [ ]:
etapas = []
f = d.copy();                                                  etapas.append(('Todos os municípios', len(f)))
f = f[f.participou_sinisa];                                    etapas.append(('Responderam o SINISA 2024', len(f)))
f = f[f.qtd_domicilios_risco > 0];                             etapas.append(('Declaram moradias em risco', len(f)))
f = f[f.criterio_fiscal_priorizacao];                          etapas.append(('Sem CAPAG A ou B', len(f)))
f = f[f.mapeamento_area_risco == 'Sim'];                       etapas.append(('Têm o risco mapeado', len(f)))
f = f[(f.municipio_critico == 'Sim') & f.tem_dano_atlas];      etapas.append(('Críticos e com dano no Atlas', len(f)))
eleg = f.copy()
assert len(eleg) == 318, 'O funil deveria terminar em 318 elegíveis'
pd.DataFrame(etapas, columns=['etapa', 'municipios'])

## 2. Ordem dos 20 e valores de referência

`ordenar` produz a lista oficial (A) e a alternativa (C). `alocar` acha, por bisseção, o valor por moradia `r` tal que a soma de `clip(r × moradias, piso, teto)` dê exatamente R$ 150 mi.

In [ ]:
def ordenar(e, opcao):
    e = e.copy()
    if opcao == 'A':
        chave = ['qtd_domicilios_risco', 'total_desastres', 'danos_humanos']
    else:  # 'C'
        e['nota'] = 0.7 * e.qtd_domicilios_risco.rank(pct=True) + 0.3 * e.pct_domicilios_risco.rank(pct=True)
        chave = ['nota', 'qtd_domicilios_risco', 'total_desastres']
    return e.sort_values(chave, ascending=False).reset_index(drop=True)

def alocar(moradias, total=150e6, piso=3e6, teto=15e6):
    lo, hi = 0.0, teto / moradias.min()
    for _ in range(200):
        r = (lo + hi) / 2
        if (r * moradias).clip(piso, teto).sum() < total: lo = r
        else: hi = r
    v = (r * moradias).clip(piso, teto).round(2)
    livre = v.index[(v > piso) & (v < teto)][0]          # resíduo de arredondamento vai para o 1º proporcional
    v.loc[livre] += round(total - v.sum(), 2)
    return v, r

def top20(opcao):
    t = ordenar(eleg, opcao).head(20).copy()
    t['posicao'] = range(1, 21)
    t['valor_referencia'], r = alocar(t.qtd_domicilios_risco)
    return t, r

A, rA = top20('A')
Cc, rC = top20('C')
for nome, t, r in [('A', A, rA), ('C', Cc, rC)]:
    assert abs(t.valor_referencia.sum() - 150e6) < 0.005
    print(f'{nome}: R$ {r:,.2f} por moradia · {int((t.valor_referencia >= 15e6 - .01).sum())} no teto · '
          f'{int((t.valor_referencia <= 3e6 + .01).sum())} no piso · soma R$ {t.valor_referencia.sum():,.2f}')
A[['posicao', 'Nome', 'UF', 'qtd_domicilios_risco', 'valor_referencia']]

## 3. Conferência dos números do painel

In [ ]:
fora = eleg[~eleg.Cod_IBGE_7.isin(A.Cod_IBGE_7)]
conf = pd.Series({
    'Moradias em risco (20)': A.qtd_domicilios_risco.sum(),
    'Pessoas afetadas 2021–25': A.danos_humanos.sum(),
    'Desastres 2021–25': A.total_desastres.sum(),
    'Danos DM + PEPL (R$)': round(A.danos_rs.sum(), 2),
    'Moradias dos que ficam de fora': fora.qtd_domicilios_risco.sum(),
    'Parte da cidade em risco, média (%)': round(A.pct_domicilios_risco.mean(), 1),
})
assert conf['Moradias em risco (20)'] == 713211 and conf['Pessoas afetadas 2021–25'] == 341914
conf

## 4. Arquivos do Looker Studio

In [ ]:
OUT = 'painel'
os.makedirs(OUT, exist_ok=True)
geo = pd.read_csv(URL_COORD).set_index('codigo_ibge')

top = pd.DataFrame({
    'posicao': A.posicao,
    'municipio': A.Nome,
    'uf': A.UF,
    'rotulo': [f'{p}. {n} ({u})' for p, n, u in zip(A.posicao, A.Nome, A.UF)],
    'municipio_uf': [f'{n} ({u})' for n, u in zip(A.Nome, A.UF)],
    'cod_ibge': A.Cod_IBGE_7,
    'capag': A.capag.fillna('sem nota'),
    'moradias_em_risco': A.qtd_domicilios_risco.astype(int),
    'parte_da_cidade_em_risco': (A.pct_domicilios_risco / 100).round(4),   # fração → tipo Percentual no Looker
    'desastres_2021_25': A.total_desastres.astype(int),
    'pessoas_afetadas_2021_25': A.danos_humanos.astype(int),
    'danos_materiais': A.soma_danos_materiais.round(2),
    'prejuizo_economico_publico': A.soma_prejuizo_publico.round(2),
    'danos_total': A.danos_rs.round(2),
    'valor_referencia': A.valor_referencia.values,
})
top['rs_por_moradia'] = (top.valor_referencia / top.moradias_em_risco).round(2)
top['regra_valor'] = np.where(top.valor_referencia >= 15e6 - .01, 'teto', np.where(top.valor_referencia <= 3e6 + .01, 'piso', 'proporcional'))
top['lat_long'] = [f"{geo.loc[c, 'latitude']},{geo.loc[c, 'longitude']}" for c in top.cod_ibge]
top.to_csv(f'{OUT}/painel_top20.csv', index=False, encoding='utf-8')

ROT = [e[0] for e in etapas] + ['Prioridade (esta lista)']
fun = pd.DataFrame({'ordem': range(1, len(ROT) + 1), 'etapa': ROT, 'municipios': [e[1] for e in etapas] + [20]})
fun['destaque'] = np.where(fun.ordem == len(ROT), 'sim', 'nao')
fun.to_csv(f'{OUT}/painel_funil.csv', index=False, encoding='utf-8')

g_in, g_out = 'Recebem (20)', f'Ficam de fora ({len(fora)})'
pd.DataFrame([('Municípios', 1, g_in, 20), ('Municípios', 1, g_out, len(fora)),
              ('Moradias em risco', 2, g_in, int(A.qtd_domicilios_risco.sum())),
              ('Moradias em risco', 2, g_out, int(fora.qtd_domicilios_risco.sum()))],
             columns=['indicador', 'ordem', 'grupo', 'valor']).to_csv(f'{OUT}/painel_dentro_fora.csv', index=False, encoding='utf-8')
top.head()

## 5. Tabelas de apoio: suplentes, alternativa C e ranking dos 318

In [ ]:
rankA, rankC = ordenar(eleg, 'A'), ordenar(eleg, 'C')

sup = rankA.iloc[20:]   # suplentes: todos os demais elegíveis, na ordem (21º ao 318º)
pd.DataFrame({'posicao': range(21, len(rankA) + 1), 'municipio': sup.Nome.values, 'uf': sup.UF.values,
              'moradias_em_risco': sup.qtd_domicilios_risco.astype(int).values,
              'parte_da_cidade_em_risco_pct': sup.pct_domicilios_risco.round(1).values}
             ).to_csv(f'{OUT}/suplentes.csv', index=False, encoding='utf-8')

cols = {'posicao': 'posicao', 'Nome': 'municipio', 'UF': 'uf', 'qtd_domicilios_risco': 'moradias_em_risco',
        'pct_domicilios_risco': 'parte_da_cidade_em_risco_pct', 'danos_humanos': 'pessoas_afetadas_2021_25',
        'danos_rs': 'danos_total', 'valor_referencia': 'valor_referencia'}
Cc[list(cols)].rename(columns=cols).to_csv(f'{OUT}/alternativa_C_top20.csv', index=False, encoding='utf-8')

sa, sc = set(A.Nome), set(Cc.Nome)
trocas = pd.concat([A[A.Nome.isin(sa - sc)].assign(situacao='sai na alternativa'),
                    Cc[Cc.Nome.isin(sc - sa)].assign(situacao='entra na alternativa')])
trocas[['situacao', 'Nome', 'UF', 'qtd_domicilios_risco', 'pct_domicilios_risco', 'valor_referencia']].rename(columns=cols)\
    .to_csv(f'{OUT}/trocas_A_C.csv', index=False, encoding='utf-8')

def resumo(t):
    return {'moradias_em_risco': t.qtd_domicilios_risco.sum(), 'pessoas_afetadas_2021_25': t.danos_humanos.sum(),
            'danos_total': round(t.danos_rs.sum(), 2), 'parte_da_cidade_em_risco_media_pct': round(t.pct_domicilios_risco.mean(), 1)}
comp = pd.DataFrame({'lista_A': resumo(A), 'alternativa_C': resumo(Cc)})
comp['diferenca'] = comp.alternativa_C - comp.lista_A
comp.to_csv(f'{OUT}/comparacao_A_C.csv', encoding='utf-8')

rk = eleg[['Cod_IBGE_7', 'Nome', 'UF', 'qtd_domicilios_risco', 'pct_domicilios_risco']].copy()
rk['posicao_A'] = rk.Cod_IBGE_7.map(dict(zip(rankA.Cod_IBGE_7, range(1, len(eleg) + 1))))
rk['posicao_C'] = rk.Cod_IBGE_7.map(dict(zip(rankC.Cod_IBGE_7, range(1, len(eleg) + 1))))
rk.sort_values('posicao_A').to_csv(f'{OUT}/ranking_318.csv', index=False, encoding='utf-8')

print('Suplentes:', len(sup), '· em comum entre A e C:', len(sa & sc), '· primeiros suplentes de A que entram na C:', sorted(set(sup.Nome[:5]) & (sc - sa)))
comp

## 6. Baixar os arquivos (no Colab)

In [ ]:
import shutil
shutil.make_archive('painel', 'zip', OUT)
try:
    from google.colab import files
    files.download('painel.zip')
except ImportError:
    print('Arquivos salvos em', os.path.abspath(OUT), ':', sorted(os.listdir(OUT)))

---

# 9. Validação independente — auditoria cruzada

**Notebook de origem:** `Notebook/Validacao_Independente_Celula4.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

# Validação independente — Célula 4 (Observatório do Clima)

**O que é:** refaz cada etapa **a partir dos arquivos brutos oficiais**, sem reaproveitar o código dos notebooks da célula, e compara o resultado com os CSVs que estão no repositório (`data/`).

**Como ler:** cada checagem imprime `✅` quando bate e `❌` quando diverge, sempre com o valor esperado e o obtido. Nada aqui altera os arquivos da célula.

**Arquivos necessários** (mesmos nomes que vieram das fontes):

| Fonte | Arquivo |
|---|---|
| SINISA 2024 | `SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx`, `SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx` |
| CAPAG (set/2026) | `capag-municipios-posicao-2026-set.xlsx` |
| CAPAG (a que está no Drive da célula) | `capag-municipios-2025.xlsx` |
| Atlas | `BD_Atlas_1991_2025_v1_1_2026_08_06_Consolidado.xlsx` |
| Repositório | `sinisa_limpo.csv`, `CAPAG-2025_Tratamento.csv`, `S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv`, `dataset_mestre_observatorio.csv` |

No Colab: suba todos para a pasta `/content` (ou ajuste os caminhos na célula abaixo). A leitura do Atlas leva cerca de 1 minuto.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

PASTA = os.environ.get('PASTA_DADOS', '/content')
def achar(nome):
    # procura o arquivo pelo nome (aceita prefixos no nome, ex.: 'abc123-arquivo.xlsx')
    c = sorted(glob.glob(os.path.join(PASTA, '**', '*' + nome), recursive=True))
    assert c, f'Arquivo não encontrado: {nome}'
    if len(c) > 1: print(f'(aviso) mais de um arquivo para {nome}, usando {c[0]}')
    return c[0]

ARQ = {
    'sinisa_form': achar('SINISA_AGUASPLUVIAIS_Informacoes_Formularios_Tecnicos.xlsx'),
    'sinisa_ind':  achar('SINISA_AGUASPLUVIAIS_Indicadores_AP2024.xlsx'),
    'capag_set':   achar('capag-municipios-posicao-2026-set.xlsx'),
    'capag_drive': achar('capag-municipios-2025.xlsx'),
    'atlas':       achar('BD_Atlas_1991_2025_v1_1_2026_08_06_Consolidado.xlsx'),
    'rep_sinisa':  achar('sinisa_limpo.csv'),
    'rep_capag':   achar('CAPAG-2025_Tratamento.csv'),
    'rep_s2id':    achar('S2iD_Desastres_Hidrologicos_Selecao_Auditada.csv'),
    'rep_mestre':  achar('dataset_mestre_observatorio.csv'),
}
for k, v in ARQ.items(): print(f'{k:12s} -> {os.path.basename(v)}')

def checa(nome, esperado, obtido):
    ok = (esperado == obtido) if not isinstance(esperado, float) else np.isclose(esperado, obtido)
    print(f"{'✅' if ok else '❌'} {nome}: esperado {esperado} | obtido {obtido}")
    return ok

## 1. SINISA — reconstrução a partir do bruto

Lemos a aba **Dados sobre Gestão de Riscos** pela **linha de códigos** do formulário (linha 11 do Excel), e não pela posição da coluna. Os campos usados vêm do glossário oficial:
- `GAP2110` = quantidade de domicílios sujeitos a risco de inundação (declarado pelo município)
- `GAP2105` = existe mapeamento de áreas de risco de inundação?
- `GAP2103` = total de domicílios urbanos
- `Critico` = município crítico (fonte: Serviço Geológico do Brasil, campo GAP2001 do glossário)

In [ ]:
bruto = pd.read_excel(ARQ['sinisa_form'], sheet_name='Dados sobre Gestão de Riscos', header=None, dtype=str)
codigos = [str(c).replace('*', '').strip() for c in bruto.iloc[10]]   # linha 11 do Excel
sin = bruto.iloc[11:].copy(); sin.columns = [c if c != 'nan' else f'col{i}' for i, c in enumerate(codigos)]
sin = sin[sin['Cod_IBGE'].str.match(r'^\d{7}$', na=False)]
sin['participa'] = sin['col12'].eq('Participante')
sin['risco'] = pd.to_numeric(sin['GAP2110'], errors='coerce')

checa('municípios no arquivo', 5570, len(sin))
checa('participantes', 4845, int(sin['participa'].sum()))
checa('declaram 0 moradias em risco', 2607, int((sin['risco'] == 0).sum()))
checa('declaram moradias em risco > 0', 2238, int((sin['risco'] > 0).sum()))
checa('total de moradias em risco no país', 2694600.0, float(sin['risco'].sum()))
print('\nMapeamento (GAP2105):', sin['GAP2105'].value_counts(dropna=False).to_dict())
print('Risco > 0 SEM mapeamento:', int(((sin['risco'] > 0) & (sin['GAP2105'] == 'Não')).sum()), 'municípios')

In [ ]:
# Indicadores (arquivo 2): Crítico, IAP0002, IFD0010, IGR0001
ind_b = pd.read_excel(ARQ['sinisa_ind'], header=None, dtype=str)
cod_ind = [str(c).strip() for c in ind_b.iloc[10]]
cod_ind[:6] = ['Cod', 'Nome', 'UF', 'Regiao', 'Capital', 'Critico']
ind = ind_b.iloc[11:].copy(); ind.columns = cod_ind
ind = ind[ind['Cod'].str.match(r'^\d{7}$', na=False)]

rep = pd.read_csv(ARQ['rep_sinisa'], dtype={'Cod_IBGE_7': str}, encoding='utf-8-sig')
x = rep.merge(sin[['Cod_IBGE', 'GAP2105', 'risco']], left_on='Cod_IBGE_7', right_on='Cod_IBGE')\
       .merge(ind[['Cod', 'Critico', 'IAP0002', 'IFD0010', 'IGR0001']], left_on='Cod_IBGE_7', right_on='Cod')
num = lambda s: pd.to_numeric(s, errors='coerce').astype(float)
pares = [('qtd_domicilios_risco', 'risco', True), ('mapeamento_area_risco', 'GAP2105', False),
         ('municipio_critico', 'Critico', False), ('pct_vias_rede_drenagem', 'IAP0002', True),
         ('investimento_percapita_drenagem', 'IFD0010', True), ('pct_domicilios_risco', 'IGR0001', True)]
print('Comparação município a município: sinisa_limpo.csv (repositório) x bruto')
for a, b, eh_num in pares:
    if eh_num: dif = (~np.isclose(num(x[a]), num(x[b]), equal_nan=True)).sum()
    else:      dif = (x[a].fillna('NA') != x[b].fillna('NA')).sum()
    checa(f'{a} divergências', 0, int(dif))

## 2. CAPAG — qual versão o repositório usou?

O arquivo do Tesouro tem várias abas. A nota final está na aba **Prévia da CAPAG**, coluna `CAPAG`. Comparamos o CSV do repositório com **as duas versões**: a que está no Drive da célula e a de setembro/2026.

In [ ]:
def capag(arq):
    p = pd.read_excel(arq, sheet_name='Prévia da CAPAG', header=2, dtype=str)
    p = p[p['Código Município Completo'].str.match(r'^\d{7}$', na=False)]
    return p.set_index('Código Município Completo')['CAPAG'].str.strip()

def data_ranking(arq):
    r = pd.read_excel(arq, sheet_name='Ranking', header=None, nrows=4, dtype=str)
    return [v for v in r.values.ravel() if isinstance(v, str) and ('Data' in v or '2026' in v)][:4]

cap_set, cap_drive = capag(ARQ['capag_set']), capag(ARQ['capag_drive'])
print('Datas na aba Ranking (Drive):', data_ranking(ARQ['capag_drive']))
print('Datas na aba Ranking (set/26):', data_ranking(ARQ['capag_set']))
print('\nDistribuição Drive :', cap_drive.value_counts().to_dict())
print('Distribuição set/26:', cap_set.value_counts().to_dict())

rc = pd.read_csv(ARQ['rep_capag'], dtype=str, encoding='utf-8-sig').set_index('cod_ibge')['capag']
checa('repositório x CAPAG do Drive (divergências)', 0, int((rc != cap_drive.reindex(rc.index)).sum()))
d = (rc != cap_set.reindex(rc.index))
checa('repositório x CAPAG set/26 (divergências)', 0, int(d.sum()))
print('\nExemplos de municípios que mudaram de nota:')
ex = pd.DataFrame({'repositorio': rc[d], 'set_2026': cap_set.reindex(rc.index)[d]})
ex = ex.join(sin.set_index('Cod_IBGE')['Nome_Mun'])
print(ex.loc[[c for c in ['3303906', '3300100', '4202404', '3156700'] if c in ex.index]])
print(pd.crosstab(ex['repositorio'], ex['set_2026']))

## 3. Atlas — recorte e danos humanos

Recorte usado pela célula: grupo **Hidrológico** (classificação da própria aba *Grupos de Desastres* do Atlas), eventos de 2021 a 2025, Brasil inteiro, aba **Atlas Valores Corrigidos**.

In [ ]:
cols = ['Protocolo_S2iD', 'Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'Cod_Cobrade', 'descricao_tipologia',
        'grupo_de_desastre', 'Cod_IBGE_Mun', 'Status', 'DH_MORTOS', 'DH_FERIDOS', 'DH_ENFERMOS', 'DH_DESABRIGADOS',
        'DH_DESALOJADOS', 'DH_DESAPARECIDOS', 'DH_total_danos_humanos_diretos', 'DM_total_danos_materiais', 'PEPL_total_publico']
atlas = pd.read_excel(ARQ['atlas'], sheet_name='Atlas Valores Corrigidos', usecols=cols, dtype=str)
atlas['data'] = pd.to_datetime(atlas['Data_Evento'], errors='coerce')
rec = atlas[(atlas['grupo_de_desastre'] == 'Hidrológico') & atlas['data'].dt.year.between(2021, 2025)].copy()
for c in cols[9:]: rec[c] = pd.to_numeric(rec[c], errors='coerce').fillna(0)

s2 = pd.read_csv(ARQ['rep_s2id'], sep=';', dtype={'Cod_IBGE_Mun': str}, encoding='utf-8-sig')
checa('registros no recorte', 9890, len(rec))
checa('mesmos protocolos no repositório', 9890, int(rec['Protocolo_S2iD'].isin(s2['Protocolo_S2iD']).sum()))
m = s2.merge(rec, on='Protocolo_S2iD', suffixes=('_rep', ''))
_ = checa('danos materiais (diferença > R$ 0,01)', 0, int((abs(m['DM_total_danos_materiais_rep'] - m['DM_total_danos_materiais']) > 0.01).sum()))

In [ ]:
seis = ['DH_MORTOS', 'DH_FERIDOS', 'DH_ENFERMOS', 'DH_DESABRIGADOS', 'DH_DESALOJADOS', 'DH_DESAPARECIDOS']
soma6 = m[seis].sum(axis=1)
checa('total oficial == soma das 6 categorias (registros que NÃO batem)', 0, int((soma6 != m['DH_total_danos_humanos_diretos']).sum()))
sem_feridos = m[[c for c in seis if c != 'DH_FERIDOS']].sum(axis=1)
checa('DH_total_calculado do repositório == soma SEM feridos (registros que NÃO batem)', 0, int((sem_feridos != m['DH_total_calculado']).sum()))
checa('registros com feridos > 0 (= as "606 inconsistências")', 606, int((m['DH_FERIDOS'] > 0).sum()))
print('\nExemplo:')
print(m.loc[m['DH_FERIDOS'] > 0, ['Nome_Municipio_rep', 'DH_FERIDOS', 'DH_total_calculado', 'DH_total_danos_humanos_diretos']].head(3).to_string())

## 4. Cruzamento — totais de controle do `dataset_mestre_observatorio.csv`

In [ ]:
mestre = pd.read_csv(ARQ['rep_mestre'], dtype={'Cod_IBGE_7': str}, encoding='utf-8-sig')
checa('linhas (1 por município)', 5570, len(mestre))
checa('código IBGE duplicado', 0, int(mestre['Cod_IBGE_7'].duplicated().sum()))
checa('soma de eventos = registros do recorte', 9890, int(mestre['total_desastres'].sum()))
checa('soma de danos materiais = soma do recorte (R$)', round(rec['DM_total_danos_materiais'].sum(), 0), round(mestre['soma_danos_materiais'].sum(), 0))
checa('soma_afetados = total oficial de danos humanos', int(rec['DH_total_danos_humanos_diretos'].sum()), int(mestre['soma_afetados'].sum()))
checa('municípios sem nota CAPAG no mestre', 0, int(mestre['capag'].isna().sum()))
print('   ->', mestre.loc[mestre['capag'].isna(), ['Nome', 'UF']].values.tolist())

## 5. Funil — reproduzindo o notebook `Transformacao-Filtros` e a versão corrigida

O filtro 4 do notebook (célula 4, linha 18) usa `.notna()`. Como a coluna tem `'Sim'`/`'Não'`, esse filtro não remove ninguém. Abaixo, as duas versões lado a lado, com as duas CAPAGs.

In [ ]:
base = sin[['Cod_IBGE', 'Nome_Mun', 'UF', 'participa', 'risco', 'GAP2105']].rename(columns={'Cod_IBGE': 'cod', 'Nome_Mun': 'nome', 'GAP2105': 'mapeado'})
base = base.merge(ind[['Cod', 'Critico']], left_on='cod', right_on='Cod')
ag = rec.groupby('Cod_IBGE_Mun').agg(eventos=('Protocolo_S2iD', 'count'))
base = base.join(ag, on='cod').fillna({'eventos': 0})
base['capag_drive'] = base['cod'].map(cap_drive); base['capag_set'] = base['cod'].map(cap_set)

def funil(capcol, filtro_mapa):
    f1 = base[base['participa']]; f2 = f1[f1['risco'] > 0]
    f3 = f2[f2[capcol].isin(['C', 'D'])]
    f4 = f3[f3['mapeado'].notna()] if filtro_mapa == 'notna (notebook)' else f3[f3['mapeado'] == 'Sim']
    top = f4.sort_values(['risco', 'eventos'], ascending=False).head(20)
    return [len(f1), len(f2), len(f3), len(f4)], top

res = {}
for capcol in ['capag_drive', 'capag_set']:
    for fm in ['notna (notebook)', "== 'Sim' (corrigido)"]:
        n, top = funil(capcol, fm); res[(capcol, fm)] = top
        print(f'{capcol:12s} | filtro4 {fm:22s} | funil {n} | moradias top20 = {int(top.risco.sum()):,}')
n, _ = funil('capag_drive', 'notna (notebook)')
_ = checa('reproduz o notebook (universo = 816)', 816, n[3])

In [ ]:
a = res[('capag_drive', 'notna (notebook)')]; b = res[('capag_drive', "== 'Sim' (corrigido)")]; c = res[('capag_set', "== 'Sim' (corrigido)")]
print('Saem do top 20 ao corrigir o filtro 4:', a.loc[~a.cod.isin(b.cod), ['nome', 'UF', 'mapeado']].values.tolist())
print('Saem ao trocar para a CAPAG set/26  :', b.loc[~b.cod.isin(c.cod), ['nome', 'UF']].values.tolist())
print('Entram com a CAPAG set/26           :', c.loc[~c.cod.isin(b.cod), ['nome', 'UF', 'capag_drive', 'capag_set']].values.tolist())
print('\nTop 20 (filtro corrigido + CAPAG set/26):')
print(c[['nome', 'UF', 'risco', 'capag_set', 'Critico', 'eventos']].reset_index(drop=True).to_string())
_ = checa('todos do top 20 são "Críticos"', 20, int((c['Critico'] == 'Sim').sum()))

---

# 10. Análise exploratória adicional CAPAG

**Notebook de origem:** `analise_exploratória.ipynb`

Células preservadas na ordem original; saídas antigas foram removidas para permitir execução limpa.

ANÁLISE EXPLORATÓRIA DO DATA CARD DA CAPAG

# 1. Conectando o Google Drive ao Colab

In [ ]:

# 1. Conectar o Google Drive ao Colab
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')

# 2. Definir o caminho do arquivo no seu Drive
caminho_arquivo = '/content/drive/MyDrive/Observatório de Clima-Célula 4./Dataset\'s/capag-municipios-2025.xlsx'



2. lendo as Primeiras Linhas do csv

In [ ]:

# 3. Carregar a aba da 'Prévia da CAPAG' lendo o cabeçalho na linha correta (header=2)
# Garantimos que o código IBGE seja lido como texto para manter os 7 dígitos
df_capag = pd.read_excel(
    caminho_arquivo,
    sheet_name='Prévia da CAPAG',
    header=2,
    dtype={'Código Município Completo': str}
)

# Remediar eventual linha de rodapé/total nula
df_capag = df_capag.dropna(subset=['Código Município Completo'])



3. VISÃO GERAL DO DATASET

In [ ]:

print("======================================================================")
print(f"• Total de Municípios (Linhas): {df_capag.shape[0]}")
print(f"• Total de Atributos (Colunas): {df_capag.shape[1]}")
print(f"• Chave Primária: 'Código Município Completo' ({df_capag['Código Município Completo'].nunique()} códigos únicos)")

4. DIAGNÓSTICO DE QUALIDADE DOS DADOS & VALORES FALTANTES/IRREGULARES

In [ ]:

print("======================================================================")
# Tratar strings de notas limpando espaços em branco
df_capag['CAPAG_limpa'] = df_capag['CAPAG'].astype(str).str.strip()

# Distribuição completa de todas as Notas
distribuicao = df_capag['CAPAG_limpa'].value_counts(dropna=False)
print("Distribuição das Notas da CAPAG no Brasil:")
print(distribuicao.to_string())

# Identificação de Municípios sem Nota (n.d. / n.e. / Nulos)
sem_nota = df_capag[df_capag['CAPAG_limpa'].isin(['n.d.', 'n.e.', 'nan', 'None'])]
print(f"\n• Municípios sem Nota Atribuída (n.d. ou n.e.): {len(sem_nota)} ({len(sem_nota)/len(df_capag)*100:.2f}%)")
print("  (Motivo: Não enviaram DCA/RGF/RREO no prazo exigido pelo Tesouro Nacional)")

In [ ]:

print("======================================================================")
# Municípios Elegíveis a Empréstimos (A+, A, B+, B)
elegiveis_credito = df_capag[df_capag['CAPAG_limpa'].isin(['A+', 'A', 'B+', 'B'])]

# Municípios Frágeis / Vedados de Tomar Empréstimos (Notas C e D)
vulneraveis_cd = df_capag[df_capag['CAPAG_limpa'].isin(['C', 'D'])]

qtd_cd = len(vulneraveis_cd)
pct_cd = (qtd_cd / len(df_capag)) * 100

print(f"• Municípios com Capacidade de Crédito (Notas A+, A, B+, B): {len(elegiveis_credito)} ({len(elegiveis_credito)/len(df_capag)*100:.2f}%)")
print(f"• Municípios em Fragilidade Fiscal (Notas C e D): {qtd_cd} ({pct_cd:.2f}%)")
print("  --> NOTA TÉCNICA PARA O CARLOS: Este grupo de 2.203 municípios possui impedimento legal")
print("      para obter garantia da União para empréstimos bancários. Portanto, necessitam 100%")
print("      de repasses a fundo perdido do Governo Federal para obras de prevenção de riscos.")

_________________Transformando em arquivos separados____________________________

In [ ]:
import pandas as pd
import zipfile
from google.colab import files

# 1. Montar a estrutura do Resumo Geral
df_resumo = pd.DataFrame([
    {"secao": "Identificacao", "indicador": "Fonte", "valor": "STN / Siconfi (Tesouro Nacional)", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Ano de Referencia", "valor": "Previa Ano-Base 2025 (Setembro/2026)", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Aba Utilizada", "valor": "Previa da CAPAG", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Granularidade", "valor": "1 linha por Municipio", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Chave Primaria", "valor": "Codigo Municipio Completo (IBGE 7 digitos)", "percentual": ""},
    {"secao": "Qualidade", "indicador": "Total de Municipios Analisados", "valor": "5.568", "percentual": "100.0%"},
    {"secao": "Qualidade", "indicador": "Municipios com Nota Atribuida (A+ ate D)", "valor": "4.747", "percentual": "85.26%"},
    {"secao": "Qualidade", "indicador": "Municipios sem Nota (n.d. / n.e.)", "valor": "820", "percentual": "14.74%"},
    {"secao": "Estrategia", "indicador": "Municipios com Capacidade de Credito (A+, A, B+, B)", "valor": "2.544", "percentual": "45.69%"},
    {"secao": "Estrategia", "indicador": "Municipios com Fragilidade Fiscal (C e D)", "valor": "2.203", "percentual": "39.57%"}
])

# 2. Montar a Distribuição das Notas CAPAG
df_distribuicao = pd.DataFrame([
    {"nota_capag": "C", "classificacao_fiscal": "Fragilidade Fiscal / Vedado Emprestimo", "qtd_municipios": 2184, "percentual_nacional": 39.22},
    {"nota_capag": "A", "classificacao_fiscal": "Boa Saude Fiscal / Elegivel a Credito", "qtd_municipios": 1190, "percentual_nacional": 21.37},
    {"nota_capag": "B", "classificacao_fiscal": "Boa Saude Fiscal / Elegivel a Credito", "qtd_municipios": 976, "percentual_nacional": 17.53},
    {"nota_capag": "n.d.", "classificacao_fiscal": "Nao Disponivel (Pendencia Contabil DCA/RGF)", "qtd_municipios": 683, "percentual_nacional": 12.27},
    {"nota_capag": "A+", "classificacao_fiscal": "Excelente Saude Fiscal / Elegivel a Credito", "qtd_municipios": 265, "percentual_nacional": 4.76},
    {"nota_capag": "n.e.", "classificacao_fiscal": "Nao Elegivel (Pendencia de Envio STN)", "qtd_municipios": 137, "percentual_nacional": 2.46},
    {"nota_capag": "B+", "classificacao_fiscal": "Boa Saude Fiscal / Elegivel a Credito", "qtd_municipios": 113, "percentual_nacional": 2.03},
    {"nota_capag": "D", "classificacao_fiscal": "Alta Fragilidade Fiscal / Vedado Emprestimo", "qtd_municipios": 19, "percentual_nacional": 0.34}
])

# 3. Montar as Limitações e Regra de Negócio
df_limitacoes = pd.DataFrame([
    {"tipo": "limitacao", "item": "820 municipios (14.74%) nao possuem nota oficial (n.d./n.e.) por descumprimento de prazos contabeis (DCA/RGF/RREO)."},
    {"tipo": "limitacao", "item": "A nota CAPAG reflete a situacao fiscal global do municipio, nao a capacidade executiva da Defesa Civil local."},
    {"tipo": "regra_de_negocio", "item": "Priorizacao de Repasses: Municipios com CAPAG C e D devem ter prioridade absoluta no recurso a fundo perdido."},
    {"tipo": "regra_de_negocio", "item": "Impedimento Legal: Prefeituras C e D sao proibidas por lei de contratar financiamentos com garantia da Uniao (ex.: BNDES/Fundo Clima)."},
    {"tipo": "regra_de_negocio", "item": "Fundamentacao para Carlos (SEDEC): O repasse nao-reembolsavel e a unica alternativa legal e financeira para viabilizar obras nessas localidades."}
])

# Gerar o arquivo Excel com 3 abas
excel_filename = "DataCard_CAPAG_2025_Oficial.xlsx"
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_resumo.to_excel(writer, sheet_name="01_Resumo_Geral", index=False)
    df_distribuicao.to_excel(writer, sheet_name="02_Distribuição_Notas", index=False)
    df_limitacoes.to_excel(writer, sheet_name="03_Limitacoes_e_Regras", index=False)

# Gerar o arquivo ZIP com CSVs
zip_filename = "DataCard_CAPAG_2025_CSV.zip"
with zipfile.ZipFile(zip_filename, 'w') as z:
    z.writestr("01_resumo_geral.csv", df_resumo.to_csv(index=False, encoding='utf-8-sig'))
    z.writestr("02_distribuicao_notas.csv", df_distribuicao.to_csv(index=False, encoding='utf-8-sig'))
    z.writestr("03_limitacoes_recomendacoes.csv", df_limitacoes.to_csv(index=False, encoding='utf-8-sig'))

# Forçar o download automático no seu navegador
files.download(excel_filename)
files.download(zip_filename)

Expontando para PDF

In [ ]:
!pip install reportlab

from google.colab import files
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_filename = "Canvas_DataCard_CAPAG_2025.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=landscape(letter),
    rightMargin=20, leftMargin=20, topMargin=20, bottomMargin=20
)

styles = getSampleStyleSheet()

# Estilos de Texto
title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=18, leading=22, textColor=colors.HexColor('#1A365D'))
subtitle_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=10, leading=12, textColor=colors.HexColor('#4A5568'))
header_box_style = ParagraphStyle('BoxHeader', parent=styles['Heading2'], fontName='Helvetica-Bold', fontSize=11, leading=13, textColor=colors.white)
body_bold_style = ParagraphStyle('BodyBold', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8.5, leading=10.5, textColor=colors.HexColor('#1A202C'))
body_style = ParagraphStyle('BodyTextCustom', parent=styles['Normal'], fontName='Helvetica', fontSize=8, leading=10, textColor=colors.HexColor('#2D3748'))
tag_c_d = ParagraphStyle('TagCD', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8, leading=10, textColor=colors.HexColor('#C53030'))
tag_a_b = ParagraphStyle('TagAB', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8, leading=10, textColor=colors.HexColor('#276749'))

# 1. CONTEÚDO RESUMO GERAL
p1_content = [
    Paragraph("<b>Origem do Dado:</b> STN / Siconfi (Tesouro Nacional)", body_style),
    Paragraph("<b>Ano-Base / Versão:</b> Prévia Ano-Base 2025 (Set/2026)", body_style),
    Paragraph("<b>Granularidade:</b> 1 linha por Município (Brasil)", body_style),
    Paragraph("<b>Chave Primária:</b> Código IBGE de 7 dígitos", body_style),
    Paragraph("<b>Total de Municípios:</b> 5.568 prefeituras", body_style),
    Spacer(1, 4),
    Paragraph("<b>Qualidade & Preenchimento:</b>", body_bold_style),
    Paragraph("• <b>Nota Atribuída (A+ a D):</b> 4.747 mun. (85,26%)", body_style),
    Paragraph("• <b>Sem Nota (n.d. / n.e.):</b> 820 mun. (14,74%)", body_style),
    Paragraph("• <b>Inconsistências/Nulos na Chave:</b> 0 (Chave IBGE limpa)", body_style),
]

# 2. CONTEÚDO TABELA DISTRIBUIÇÃO
table_data = [
    [Paragraph("<b>Nota</b>", body_bold_style), Paragraph("<b>Perfil Fiscal</b>", body_bold_style), Paragraph("<b>Qtd</b>", body_bold_style), Paragraph("<b>%</b>", body_bold_style)],
    [Paragraph("C", tag_c_d), Paragraph("Fragilidade Fiscal / Vedado Crédito", body_style), Paragraph("2.184", body_style), Paragraph("39,22%", body_style)],
    [Paragraph("A", tag_a_b), Paragraph("Boa Saúde Fiscal / Elegível Crédito", body_style), Paragraph("1.190", body_style), Paragraph("21,37%", body_style)],
    [Paragraph("B", tag_a_b), Paragraph("Boa Saúde Fiscal / Elegível Crédito", body_style), Paragraph("976", body_style), Paragraph("17,53%", body_style)],
    [Paragraph("n.d.", body_style), Paragraph("Pendência Contábil (DCA/RGF)", body_style), Paragraph("683", body_style), Paragraph("12,27%", body_style)],
    [Paragraph("A+", tag_a_b), Paragraph("Excelente Saúde Fiscal", body_style), Paragraph("265", body_style), Paragraph("4,76%", body_style)],
    [Paragraph("n.e.", body_style), Paragraph("Não Elegível (Pendência STN)", body_style), Paragraph("137", body_style), Paragraph("2,46%", body_style)],
    [Paragraph("B+", tag_a_b), Paragraph("Boa Saúde Fiscal / Elegível Crédito", body_style), Paragraph("113", body_style), Paragraph("2,03%", body_style)],
    [Paragraph("D", tag_c_d), Paragraph("Alta Fragilidade Fiscal / Vedado Crédito", body_style), Paragraph("19", body_style), Paragraph("0,34%", body_style)],
]
t_dist = Table(table_data, colWidths=[28, 145, 32, 35])
t_dist.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#EDF2F7')),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CBD5E0')),
    ('TOPPADDING', (0,0), (-1,-1), 2), ('BOTTOMPADDING', (0,0), (-1,-1), 2),
]))
p2_content = [t_dist]

# 3. CONTEÚDO LIMITAÇÕES E RECOMENDAÇÕES
p3_content = [
    Paragraph("<b>1. Limitações dos Dados:</b>", body_bold_style),
    Paragraph("• <b>820 municípios (14,74%)</b> figuram como <i>n.d.</i> ou <i>n.e.</i> por descumprimento de prazos no envio das declarações contábeis (DCA/RGF/RREO) ao Siconfi/STN.", body_style),
    Paragraph("• A nota CAPAG reflete o risco fiscal global do ente municipal, e não a capacidade técnica/operacional da Defesa Civil local.", body_style),
    Spacer(1, 4),
    Paragraph("<b>2. Regra de Negócio & Recomendação (Carlos):</b>", body_bold_style),
    Paragraph("• <b>Prioridade Absoluta (CAPAG C e D):</b> Os <b>2.203 municípios (39,57%)</b> com notas C e D possuem impedimento legal de obter garantia da União para empréstimos bancários (ex.: BNDES / Fundo Clima).", body_style),
    Paragraph("• <b>Justificativa Técnica:</b> O repasse a fundo perdido (Sedec/MIDR) é a <b>única alternativa financeira legal</b> para viabilizar obras de prevenção contra inundações nessas cidades.", body_style),
]

# Função para construir as caixas do Canvas
def make_box(header_title, content_list, header_bg_color, width):
    header_table = Table([[Paragraph(header_title, header_box_style)]], colWidths=[width])
    header_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor(header_bg_color)),
        ('TOPPADDING', (0,0), (-1,-1), 5), ('BOTTOMPADDING', (0,0), (-1,-1), 5), ('LEFTPADDING', (0,0), (-1,-1), 8),
    ]))
    body_table = Table([[content_list]], colWidths=[width])
    body_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#FAFAFA')),
        ('BOX', (0,0), (-1,-1), 1, colors.HexColor('#E2E8F0')),
        ('TOPPADDING', (0,0), (-1,-1), 8), ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LEFTPADDING', (0,0), (-1,-1), 8), ('RIGHTPADDING', (0,0), (-1,-1), 8),
    ]))
    return Table([[header_table], [body_table]], colWidths=[width])

# Montagem do Grid Horizontal do Canvas
col1 = make_box("1. RESUMO GERAL", p1_content, "#2B6CB0", 240)
col2 = make_box("2. DISTRIBUIÇÃO DA CAPAG", p2_content, "#2C7A7B", 250)
col3 = make_box("3. LIMITAÇÕES E RECOMENDAÇÕES", p3_content, "#C53030", 250)

canvas_grid = Table([[col1, col2, col3]], colWidths=[245, 255, 255])
canvas_grid.setStyle(TableStyle([('VALIGN', (0,0), (-1,-1), 'TOP')]))

elements = [
    Paragraph("Canvas do Data Card — Capacidade de Pagamento (CAPAG 2025)", title_style),
    Paragraph("<b>Projeto:</b> Observatório do Clima | <b>Sprint 2 — Vaga 2.2</b> | <b>Persona:</b> Carlos (SEDEC/MIDR)", subtitle_style),
    Spacer(1, 8),
    HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#CBD5E0'), spaceAfter=10),
    canvas_grid
]

doc.build(elements)
files.download(pdf_filename)